# Case Assignment Round Robin

In [1]:
import numpy as np
import pandas as pd
import os

import win32com.client
from win32com.client import constants

from datetime import date
from datetime import datetime
import random

from shutil import copyfile


# Reminder

Please update the CM List in the file from KYC Ops.<br>
The correct CM List is in the file <b>CM List.xlsx</b>.<br>
Please update any new CM or changes in CM Team memeber / CM Team Head in this file.<br>

Please Copy the following Columns to Case Assignment File from KYC Ops from the result generated from Python:
1. WIP Mater Group CM
2. Master Group ID
3. Master Group
4. IMIS Group No
5. IMIS Group
6. Assign by TH
7. CM Name    

## Post Assignment Checking
Please perform the following checking after major batch case assignment:
1. Check WIP Mater Group CM vs CM Name
2. Check the file "Assigned MG without T0" to ensure no duplicated case

## Logic of Case Assignment
Currently, the case assignment are with the following requirement:<br>
1. Case is assigned based on FTE of each Team;
2. Forced Assignment List is overriding all case assigment logic;
3. MGCSM cases are handled by MGCSM;
4. MGCSM cases can be assigned to ETB Team by using the Forced Assignment List;
5. BBRM Client with the same IMIS Group ID is to be with the same CM;
6. CORP Client with the same Master Group ID is to be assigned to same CM Team;
7. Assuming only Corp and BBRM case will have Master Group ID and IMIS Group ID respectively.
8. After case assignment, a file will be generated for storing assigned case but T0 has not been issued. The file is for next case assignment.

# Define Parameter and File


### WIP_Day
* Define whether case (non to be triggered) is treated as WIP. 
* For example, if a case is completed at T60 and a new case within the same Master Group  / IMIS Group is triggered, it will have a better sutomer experience to be handled by the same CM.
* Setting a day for WIP_Day is to ensure the Group of customer can be handled by other CM / CM Team for fariness.

### SkippedTeam
* MGCSM must be in the array.
* Please add the Team whose is to be skip "normal" case assigment.

### The Rest
* The rest are necessary data sources for case assignment.

In [2]:
MGCSM_TH = 'Ken Y K FONG'
RCSM_TH = 'Cherry H Y MAK'

In [3]:
PreCA_CutDay = 90 # For Assigned Case but T0 end up not issued
WIP_Day = 180 # For cases with T0

CaseAssignFile = 'CM Assignment 17Aug2026' # Please exclude Extension
HorisFile = 'horis_mg_202606.xlsx' # from enablement team's e-mail
MasterMI_Template = 'GS CDD FullList Master template Full - 2026-08-17.xlsx' # MI\Central Team Tool\Case Assignment share drive
MasterGp_List = 'MG pairing - Apr 2026.xlsx' # downlaod by using SAS or MI\Central Team Tool\Case Assignment share drive
FTEFile = 'ETB FTE Record.xlsx' # GS CDD\Case Assignment\GS - Periodic review share drive
ForcedCaseAssigment_File = 'Forced Case Assignment List.xlsx' # GS CDD\Case Assignment\GS - Periodic review share drive
IMIS_File = 'BBRM_IMIS_GROUP_20260807.xlsx' # MI\Central Team Tool\Case Assignment share drive
CMFullList_File = 'CM List.xlsx' # MI\Central Team Tool\Case Assignment share drive
CarryForward_File = 'Assigned MG without T0'
# BoW_File = '2021 FY BoW v0.6.7 - 2021 1Q final.xlsx'










KYC_CM_Exist = False

BRM_File = 'RM List for BRM Case.xlsx' # MI\Central Team Tool\Case Assignment share drive
KYCOps_Corp_File = 'KYC Ops Corp Case.xlsx' # MI\Central Team Tool\Case Assignment share drive
KYCOpsCMTeam = ''

CorpSubTeamMapping = 'Corp Sub Team Manual Mapping.xlsx'

RMSubTeam_File = 'cdd_review_dt_wkly_20210930.csv' # downlaod by using SAS or MI for GSCDD share drive
#KYCOps_CMTH = 'KYC OPS'

#RelatID_FName = 'BAU Cust Info RM (CI to CIN)-20191218.xlsx'

SkippedTeam = ['MARCO Y O CHUNG', 'Vasuda VENKATADRI', MGCSM_TH, RCSM_TH] #By default as this team only handle specific list of client.

SLoc = 0
iCM = 0

In [4]:
nonExist_Team = ['Alex K F Li',
'Coni C T Leung',
'Galie K Y LEE',
'Helen H L Leung',
'Karl K W Man',
'Lorita V C Chan',
'Marco Y O Chung',
'Mary L N Cheung',
'Peggy C Y Fan']

# The below is for checking existance of file
If the file does not exist, please double check the file name in the cell above.<br>
If the code below finds any missin file, the whole program will NOT be stopped.

In [5]:
def Convert_Excel_Remove_PW(ExcelFName, path):

    xlxs_FName = ExcelFName.split('.')[0] + '.xlsx' 
    
    pw_str = 'AMHCMa'
    
    xcl = win32com.client.Dispatch('Excel.Application')
    xcl.visible = True
    xlsm = xcl.workbooks.open(path + ExcelFName, 0, False, None, pw_str)
    xlsm.Unprotect(pw_str)
#     try:
#         xlsm.Unprotect(pw_str)
#     except:
#         xlsm = xcl.workbooks.open(path + ExcelFName, 0, False, None)
#         pass
    xcl.DisplayAlerts = False
#     print(xlsm.Name)
    for aSh in xlsm.Worksheets:
        try:
            aSh.AutoFilter.ShowAllData()
            print('Show All for %s' % aSh.Name)
        except:
            pass
    
    xlsm.Save()
    xlsm.SaveAs(path + xlxs_FName.split('.')[0], 51, '')
    xcl.DisplayAlerts = True
    xcl.Quit()
    return 0

In [6]:
arr = os.listdir()
path = os.getcwd() + '\\'
FName = CaseAssignFile.split('.')[0] + '.xlsm'
CaseAssignFile = CaseAssignFile.split('.')[0] + '.xlsx'
# iCM = 0

In [7]:
FileName = ''
FileName_Amend = ''
FileName_Admin = ''
FileName_FDD = ''

for aFN in arr:
    Trgt = 'CM Assignment'
    if (Trgt == aFN[:len(Trgt)]) & (('.xlsm' == aFN[-5:]) | ('.xlsx' == aFN[-5:])):
        if 'Company Amendment' in (aFN):
            FileName_Amend = aFN
            print('File is ready:\t%s' % aFN)
        elif 'Admin' in (aFN):
            FileName_Admin = aFN
            print('File is ready:\t%s' % aFN)
        elif 'FDD' in (aFN):
            FileName_FDD = aFN
            print('File is ready:\t%s' % aFN)
        else:
            FileName = aFN
            print('File is ready:\t%s' % aFN)

# if not CaseAssignFile in arr:
#     if not FName in arr:
#         print('Please check the file:\t%s' % CaseAssignFile.split('.')[0])
#     else:
#         print('File is ready:\t%s' % CaseAssignFile)
# else:
#     print('File is ready:\t%s' % CaseAssignFile)
    
if HorisFile in arr:
    print('File is ready:\t%s' % HorisFile)
else:
    print('Please check the file:\t%s' % HorisFile)
    
if MasterMI_Template in arr:
    print('File is ready:\t%s' % MasterMI_Template)
else:
    print('Please check the file:\t%s' % MasterMI_Template)
    
if MasterGp_List in arr:
    print('File is ready:\t%s' % MasterGp_List)
else:
    print('Please check the file:\t%s' % MasterGp_List)
    
if FTEFile in arr:
    print('File is ready:\t%s' % FTEFile)
else:
    print('Please check the file:\t%s' % FTEFile)
    
if ForcedCaseAssigment_File in arr:
    print('File is ready:\t%s' % ForcedCaseAssigment_File)
else:
    print('Please check the file:\t%s' % ForcedCaseAssigment_File)
    
if IMIS_File in arr:
    print('File is ready:\t%s' % IMIS_File)
else:
    print('Please check the file:\t%s' % IMIS_File)

if CMFullList_File in arr:
    print('File is ready:\t%s' % CMFullList_File)
else:
    print('Please check the file:\t%s' % CMFullList_File)
    
if BRM_File in arr:
    print('File is ready:\t%s' % BRM_File)
else:
    print('Please check the file:\t%s' % BRM_File)
    
if RMSubTeam_File in arr:
    print('File is ready:\t%s' % RMSubTeam_File)
else:
    print('Please check the file:\t%s' % RMSubTeam_File)
    
if KYCOps_Corp_File in arr:
    print('File is ready:\t%s' % KYCOps_Corp_File)
else:
    print('Please check the file:\t%s' % KYCOps_Corp_File)

for aFile in arr:
    Found = 0
    if CarryForward_File in aFile:
        print('File is ready:\t%s' % CarryForward_File)
        Found = 1
        break
if Found == 0:
    print('Please check the file:\t%s' % CarryForward_File)

File is ready:	CM Assignment 17Aug2026.xlsm
File is ready:	horis_mg_202606.xlsx
File is ready:	GS CDD FullList Master template Full - 2026-08-17.xlsx
File is ready:	MG pairing - Apr 2026.xlsx
File is ready:	ETB FTE Record.xlsx
File is ready:	Forced Case Assignment List.xlsx
File is ready:	BBRM_IMIS_GROUP_20260807.xlsx
File is ready:	CM List.xlsx
Please check the file:	RM List for BRM Case.xlsx
Please check the file:	cdd_review_dt_wkly_20210930.csv
Please check the file:	KYC Ops Corp Case.xlsx
File is ready:	Assigned MG without T0


In [8]:
Convert_Excel_Remove_PW(FName, path)

Show All for CMAssignmentFullList
Show All for CM list
Show All for Team Mapping Table
Show All for List of MSC


0

In [9]:
if FileName_Amend != '':
    Convert_Excel_Remove_PW(FileName_Amend, path)

In [10]:
if FileName_Admin != '':
    Convert_Excel_Remove_PW(FileName_Admin, path)

In [11]:
if FileName_FDD != '':
    Convert_Excel_Remove_PW(FileName_FDD, path)

# Define Function

In [12]:
# To produce a list of Team to be skipped case assignment
# Skipped Team still be assigned with case based on previous assignment and WIP, for the case with the same Customer Group.
def SkipTeam(THName, CMFullList):
    aList = []

    for aTH in THName:
        for TH in np.unique(CMFullList['CMTH']).tolist():
            if aTH.upper() in TH.upper():
                aList.append(TH)
    return aList

In [13]:
# Random Sequence for Case Assignment (CM Team)
def indexMapper_old(size):
    OrgID = list(range(0, size))
    Sid = int(random()*size)
    for i in range(0, size):
        if OrgID[i] == Sid:
            OrgID[i] = 0
        elif OrgID[i] < Sid:
            OrgID[i] = size - Sid + OrgID[i]
        else:
            OrgID[i] = OrgID[i] - Sid
            
    return OrgID

def indexMapper(size):
    OrgID = list(range(0, size))
    
    return random.sample(OrgID, size)


In [14]:
# generate a Sequency of CMTH / provide a CMTH for Case Assignment 
def THSeqForAssg(CMTeam, nC, ColN, Overload, shift=0):
    AssList = []
    nAC = 0
    i = 0
    CaseN = CMTeam[ColN].tolist()
    #print(CaseN)
    if Overload:
        nC_Tmp = nC
        
        while nC_Tmp > 0:
            
            if CMTeam.iloc[(i + shift) % CMTeam.shape[0]]['Skipped']:
                pass
            else:
                TH = CMTeam.index[(i + shift) % CMTeam.shape[0]]

                AssList.append(TH)
                CaseN[(i + shift) % CMTeam.shape[0]] = CaseN[i % CMTeam.shape[0]] - 1
                nC_Tmp -= 1
            i = i+1
        pass
    else:
        while nAC < nC:
            #print(i + shift)
            #print(CMTeam.iloc[(i + shift) % CMTeam.shape[0]]['Skipped'])
            if CMTeam.iloc[(i + shift) % CMTeam.shape[0]]['Skipped']:
                pass
            else:
                
                TH = CMTeam.index[(i + shift) % CMTeam.shape[0]]
                #print(TH)
                #print(CaseN[(i + shift) % CMTeam.shape[0]])
                if CaseN[(i + shift) % CMTeam.shape[0]] > 0:
                    AssList.append(TH)
                    CaseN[(i + shift) % CMTeam.shape[0]] = CaseN[(i + shift) % CMTeam.shape[0]] - 1
                    nAC += 1
            i = i+1
    NLoc = CMTeam.iloc[CMTeam.index == AssList[len(AssList)-1]]['Assign Seq'].values + 1
    #print(NLoc)
    #print(CMTeam['Assign Seq'][CMTeam.index == AssList[len(AssList)-1]])
    if NLoc >= CMTeam.shape[0]:
        NLoc = [0]
        pass
    
    return AssList, NLoc[0]

In [15]:
# Return a list of boolean for filtering

def WIPCase(StL1):

    CS = StL1[0]
    # Master Status L1 Status (Short)
    WIP = ['B', 'C', 'D', 'E', 'F']
    #WIP = ['B', 'C', 'D', 'E', 'F', 'X', 'Z']
    if CS in WIP:
        return True
    else:
        return False

def CompletedCase(StL1):
    CS = StL1[0]
    WIP = ['G']
    if CS in WIP:
        return True
    else:
        return False

In [16]:
# For update Capacity for Case Assignment Process or post asosgnment MI

def UpdateCapacity(CMTeam, CAListMst):
#     print(CAListMst)
    CAListMst['Assign by TH'] = CAListMst['Assign by TH'].str.upper()
    Sel01 = CAListMst['Assign by TH'] != ''
    Sel02 = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel03 = CAListMst['CMB_Segment_S1'] == 'CORP'
    Sel04 = CAListMst['CMB_Segment_S1'] == 'CMB'

    #Assigned = CAListMst[np.logical_and(Sel01, Sel02)]['Assign by TH'].value_counts()
    Assigned = pd.pivot_table(CAListMst[(Sel01 & Sel02)], index=['Assign by TH'], values=['Customer_ID_S1'] , aggfunc='count', fill_value=0)
#     print(Assigned)
    
    CMFL = CMTeam.index.tolist()
#     print(CMFL)
    T_Type = 'Total No of ' + str('BBRM') + ' Assigned'
    
    
    for aTH in CMFL:
        if aTH in Assigned.index.tolist():
            CMTeam.loc[aTH, T_Type] = Assigned.loc[aTH]['Customer_ID_S1']
    
    #Assigned = CAListMst[np.logical_and(Sel01, Sel03)]['Assign by TH'].value_counts()
    Assigned = pd.pivot_table(CAListMst[np.logical_and(Sel01, Sel03)], index=['Assign by TH'], values=['Customer_ID_S1'] , aggfunc='count', fill_value=0)
#     print(Assigned)
    
    CMFL = CMTeam.index.tolist()
    T_Type = 'Total No of ' + str('Corp') + ' Assigned'
    
    for aTH in CMFL:
        if aTH in Assigned.index.tolist():
            CMTeam.loc[aTH, T_Type] = Assigned.loc[aTH]['Customer_ID_S1']
    
    Assigned = pd.pivot_table(CAListMst[np.logical_and(Sel01, Sel04)], index=['Assign by TH'], values=['Customer_ID_S1'] , aggfunc='count', fill_value=0)
#     print(Assigned)
    
    CMFL = CMTeam.index.tolist()
    T_Type = 'Total No of ' + str('CMB') + ' Assigned'
    
    
    for aTH in CMFL:
        if aTH in Assigned.index.tolist():
            CMTeam.loc[aTH, T_Type] = Assigned.loc[aTH]['Customer_ID_S1']
            
            
    CMTeam['Total No of CMB Assigned'] = CMTeam['Total No of CMB Assigned'].fillna(0)
    CMTeam['Total No of Assigned'] = CMTeam['Total No of BBRM Assigned'] + CMTeam['Total No of Corp Assigned'] + CMTeam['Total No of CMB Assigned']
    CMTeam['Capacity'] = CMTeam['No of Total Case'] - CMTeam['Total No of Assigned']
    return CMTeam


In [17]:
# To get the specific Assignment Sequence from a Case Assignment Capacity File

def GetOldImportSeq(CMTeam, Old_CapacityFile):
    CMTeam_Old = pd.read_excel(Old_CapacityFile, index_col=0)
    OldSeq = CMTeam_Old['Assign Seq']
    
    for aCM in CMTeam_Old.index.tolist():
        CMTeam.loc[aCM, 'Assign Seq'] = OldSeq[aCM]
    
    return CMTeam

In [18]:
# To modify CM Team Head Name to a proper format

def ModCMTHIndex(CMTeam):
    
    CList = CMTeam.index.tolist()
    
    
    for i in range(0, len(CList)):
        
        tmp = CList[i]
        
        
        tmp = tmp.upper()
        tmp = " ".join(tmp.split())
        CList[i] = tmp
    
    CMTeam.index = CList
    
    return CMTeam

def ModCMTH(aTH):
    
    if 'str' in str(type(aTH)): 
        aTH = aTH.upper()
        aTH = " ".join(aTH.split())
    
    #for i in range(0, len(CList)):
    #    tmp = CList[i]
    #    tmp = tmp.upper()
    #    tmp = " ".join(tmp.split())
    #    CList[i] = tmp
    
    return aTH

In [19]:
# Map CM Team by using the CM e-mail Name

def MapCMTeam(aCM, CMAll):
    Sel = CMAll['Case Manager Email'] == aCM
    if np.sum(Sel) > 0:
        return CMAll[Sel]['TH email'].values[0]
    else:
        return 'Not Match'

In [20]:
# Function for Development Stage
# Not in use now


def GetCaseCMTHRMTeam(CMTH, RMTeam, CAListMst):
    if CMTH == '':
        Sel01 = np.repeat(True, CAListMst.shape[0])
        pass
    else:
        Sel01 = CAListMst['Assign by TH'] == CMTH
    Sel02 = CAListMst['RM_Team_Group_S1'] == RMTeam
    Sel = np.logical_and(Sel01, Sel02)
    tmpDF = CAListMst[Sel02][['Customer_ID_S1', 'Master Group ID', 'Master Group']]
    return tmpDF

In [21]:
# Function for Development Stage
# Not in use now


def GetCaseCMTHRMTeam(CMTH, RMTeam, CAListMst):
    if CMTH == '':
        Sel01 = np.repeat(True, CAListMst.shape[0])
        pass
    else:
        Sel01 = CAListMst['Assign by TH'] == CMTH
    Sel02 = CAListMst['RM_Team_Group_S1'] == RMTeam
    Sel = np.logical_and(Sel01, Sel02)
    tmpDF = CAListMst[Sel02][['Customer_ID_S1', 'Master Group ID', 'Master Group']]
    return tmpDF

In [22]:
# Get the latest file when file name with suffix in yyyymmdd format

def getFormatFile(prefix, sep, suffix, op=0):
    maxD = 0
    prefix = str(prefix)
    sep = str(sep)
    suffix = str(suffix)
    for filename in os.listdir('./'):
        #print(filename)

        if filename[:len(prefix)] == prefix:
            tMax = int(filename.split(sep)[1].split('.')[0])
            #print(tMax)
            if tMax > maxD:
                maxD = tMax
    filename = prefix + sep + str(maxD) + suffix
    print('File found:\t%s' % filename)
    if op != 0:
        return pd.ExcelFile('./' + filename)
    else:
        return filename

In [23]:
# To select 'WIP' case from Master MI Template and Days from T0

def WIPSelectot(WIPMI_T, CDays):
    WIPMI = WIPMI_T.copy(deep=True)
    WIPMI = WIPMI[~WIPMI['Master-CM Team Head'].isin(nonExist_Team)]
    if CDays > 0:
        SelT0 = (date.today() - (WIPMI['T0'].apply(lambda x: x.date()))).apply(lambda x: x.days) < CDays
        print('Total no of cases:\t%s' % str(np.sum(SelT0)))
        pctR = 100 * np.sum(SelT0) / WIPMI_T.shape[0]
        print('Pct of total:\t%.2f%%' % pctR)
        WIPMI = WIPMI[SelT0]
        return WIPMI
    else:
        print('Total no of cases:\t' + str(WIPMI_T.shape[0]))
        return WIPMI_T
    


In [24]:
# To prepare DF of 'WIP' BBRM cases with Customer Group features and previous CM 

def PrepareWIPRelIDProfile(WIPMI):
    WIPMI = WIPMI[WIPMI['Segment'] == 'BBRM']
    
    WIPMI = WIPMI.sort_values(by=['T0'], ascending=False)
    #print(WIPMI.shape)
    WIPMI = WIPMI.drop_duplicates(['IMISGroupNo'], keep='first')
    #print(WIPMI.shape)
    #WIPMI = WIPMI[WIPMI['MasterGroup ID'] != 1514726659]
    #print(WIPMI.shape)
    WIPMI = WIPMI[np.logical_not(WIPMI['IMISGroupNo'].isnull())]
    #print(WIPMI.shape)
    WIPMI = WIPMI.sort_values(by=['Master-CM Team Head'])
    return WIPMI[['IMISGroupNo', 'IMIS Group Name',  'Master-CM Team Head', 'Master-CM Name', 'T0']]

In [25]:
# To prepare DF of 'WIP' CORP cases with Customer Group features and previous CM 

def PrepareWIPMGProfile(WIPMI):
    WIPMI = WIPMI[(WIPMI['Segment'] == 'CORP') | (WIPMI['Segment'] == 'CMB')]
    WIPMI = WIPMI.sort_values(by=['T0'], ascending=False)
    #print(WIPMI.shape)
    WIPMI = WIPMI.drop_duplicates(['MasterGroup ID'], keep='first')
    #print(WIPMI.shape)
    WIPMI = WIPMI[WIPMI['MasterGroup ID'] != 1514726659]
    #print(WIPMI.shape)
    WIPMI = WIPMI[np.logical_not(WIPMI['MasterGroup ID'].isnull())]
    #print(WIPMI.shape)
    WIPMI = WIPMI.sort_values(by=['Master-CM Team Head'])
    return WIPMI[['MasterGroup ID', 'MasterGroup Name',  'Master-CM Team Head', 'Master-CM Name', 'T0']]



In [26]:
# Convert CM e-mail name to Firat and last Name

def GetMGCM_Name(aName):
    if '/' in aName:
        return aName
    aName = aName.strip()
    CMName = aName.split(" ")
    
    CMName = CMName[0] + " " + CMName[len(CMName) - 1]
    
    return CMName

In [27]:
# To check if the Capacity will be negative after new case is assigned

def CheckCapacity(numGpCase, CMTH, CMTeam):
    #print(numGpCase)
    #print(CMTeam.loc[CMTH, 'Capacity'] >= numGpCase)
    if CMTeam.loc[CMTH, 'Capacity'] >= numGpCase:
        return 0
    else:
        return 1

# To Locate the Team which can accept the batch of cases without overloading

def LocateMaxCapacity(numGpCase, CMTeam):
    tmpDF = CMTeam[['Capacity']]
    #print(tmpDF)
    tmpDF['Capacity'] = tmpDF['Capacity'] - numGpCase
    #print(tmpDF.idxmax(axis = 0)['Capacity'])
    
    maxCMTH = tmpDF.idxmax(axis = 0)['Capacity']
    maxCap = tmpDF.loc[maxCMTH, 'Capacity']
    #print("%s:\t%d" % (maxCMTH, maxCap))
    if maxCap >= 0:
        return maxCMTH
    else:
        return 0
    
    

# Initiation

File Data Source:<br>
1. horis;
2. GS CDD FullList Master template - For Case Assignment;
3. Case Assignment File (from KYC Ops)

In [28]:
# Read Data for KYC Ops Review Team Assinment Logic

if KYC_CM_Exist:
    BRM = pd.read_excel(io=BRM_File)
#     RMSubTeam = pd.read_excel(io=RMSubTeam_File)
    RMSubTeam = pd.read_csv(RMSubTeam_File)
    RMSubTeamCorp_Manual = pd.read_excel(CorpSubTeamMapping)

In [29]:
# Read BBRM IMIS Group Data

IMIS = pd.read_excel(io=IMIS_File)
IMIS = IMIS.dropna(subset=['Group Name'])

In [30]:
# Read and Prepare horis file

MGFL = pd.read_excel(HorisFile)
MGFL = MGFL[['CIN', 'Master_Group_ID', 'Master_Group_Name', 'Customer_Name']]
# MGFL = MGFL[['Real_Customer_ID', 'Master_Group_ID', 'Master_Group_Name', 'Customer_Name']]
MGFL = MGFL[MGFL['Master_Group_ID'] != 1514726659]
MGFL = MGFL[MGFL['Master_Group_ID'] != 1514726667]
# print('0')


In [31]:
######## True Code ########
# Read and Prepare WIP Data
WIPMI = pd.read_excel(MasterMI_Template)
WIPMI_C = WIPMI.copy(deep=True)
WIPMI['Master-CM Team Head'] = WIPMI['Master-CM Team Head'].replace(0, '')
WIPMI['Master-CM Team Head'] = WIPMI['Master-CM Team Head'].apply(lambda x: ModCMTH(x))
WIPMI['Master-CM Name'] = WIPMI['Master-CM Name'].apply(lambda x: ModCMTH(x))
ComMI = WIPMI.copy(deep=True)

WIPMI['Master Status L1'] = WIPMI['Master Status L1'].fillna('A - ')
WIPMI = WIPMI[WIPMI['Master Status L1'].apply(lambda x: WIPCase(x))]



WIPMI.loc[WIPMI['MasterGroup Name']=='CMB Hong Kong Unassigned Customers - LC', 'MasterGroup Name'] = np.nan
WIPMI.loc[WIPMI['MasterGroup ID']==1514726659, 'MasterGroup Name'] = np.nan
WIPMI.loc[:, 'MasterGroup ID'] = WIPMI['MasterGroup ID'].fillna('')
WIPMI.loc[:, 'IMISGroupNo'] = WIPMI['IMISGroupNo'].fillna('')
#ComMI = ComMI[ComMI['Master Status L1'].apply(lambda x: CompletedCase(x))]


WIPMI_T = WIPMI.copy(deep=True)

WIPMI = WIPSelectot(WIPMI_T, WIP_Day)
WIPMIMG_P = PrepareWIPMGProfile(WIPMI)
WIPMIRelID_P = PrepareWIPRelIDProfile(WIPMI)

Total no of cases:	4389
Pct of total:	96.61%


In [32]:
CaseAssignFile = FName.split('.')[0] + '.xlsx'
CMAFile = pd.ExcelFile(CaseAssignFile)
CAListMst = pd.read_excel(CMAFile, 'CMAssignmentFullList')
ColCAListMst = CAListMst.columns.tolist()
try:
    CognosRecord = pd.read_excel(CMAFile, 'Cognos')
    CognosRecord = CognosRecord[['Real Customer ID 1', 'CIBM/COB/BUB team']]
    CAListMst_Main = CAListMst.merge(CognosRecord, left_on='Customer_ID_S1', right_on='Real Customer ID 1', how='left')
except:
    CognosRecord = pd.DataFrame()
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


In [33]:
if FileName_Amend != '':
    CaseAssignFile = FileName_Amend.split('.')[0] + '.xlsx'
    CMAFile = pd.ExcelFile(CaseAssignFile)
    CAListMst = pd.read_excel(CMAFile, 'CMAssignmentFullList')

    try:
        CognosRecord = pd.read_excel(CMAFile, 'Cognos')
        CognosRecord = CognosRecord[['Real Customer ID 1', 'CIBM/COB/BUB team']]
        CAListMst_Amend = CAListMst.merge(CognosRecord, left_on='Customer_ID_S1', right_on='Real Customer ID 1', how='left')
    except:
        CognosRecord = pd.DataFrame()
    print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

In [34]:
if FileName_Admin != '':
    CaseAssignFile = FileName_Admin.split('.')[0] + '.xlsx'
    CMAFile = pd.ExcelFile(CaseAssignFile)
    CAListMst = pd.read_excel(CMAFile, 'CMAssignmentFullList')

    try:
        CognosRecord = pd.read_excel(CMAFile, 'Cognos')
        CognosRecord = CognosRecord[['Real Customer ID 1', 'CIBM/COB/BUB team']]
        CAListMst_Admin = CAListMst.merge(CognosRecord, left_on='Customer_ID_S1', right_on='Real Customer ID 1', how='left')
    except:
        CognosRecord = pd.DataFrame()
    print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

In [35]:
if FileName_FDD != '':
    CaseAssignFile = FileName_FDD.split('.')[0] + '.xlsx'
    CMAFile = pd.ExcelFile(CaseAssignFile)
    CAListMst = pd.read_excel(CMAFile, 'CMAssignmentFullList')

    try:
        CognosRecord = pd.read_excel(CMAFile, 'Cognos')
        CognosRecord = CognosRecord[['Real Customer ID 1', 'CIBM/COB/BUB team']]
        CAListMst_FDD = CAListMst.merge(CognosRecord, left_on='Customer_ID_S1', right_on='Real Customer ID 1', how='left')
    except:
        CognosRecord = pd.DataFrame()
    print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

In [36]:
CAListMst = CAListMst_Main.copy(deep=True)

if FileName_Amend != '':
    CAListMst = CAListMst.append(CAListMst_Amend, ignore_index=True)
    
if FileName_Admin != '':
    CAListMst = CAListMst.append(CAListMst_Admin, ignore_index=True)
    
if FileName_FDD != '':
    CAListMst = CAListMst.append(CAListMst_FDD, ignore_index=True)

In [37]:
####### Read Case Assignment File and Map all necessary data such as Master Group ID, Relationship ID, etc

####### These should be commented - Start #######
# CMAFile = pd.ExcelFile(CaseAssignFile)
# CAListMst = pd.read_excel(CMAFile, 'CMAssignmentFullList')

# try:
#     CognosRecord = pd.read_excel(CMAFile, 'Cognos')
#     CognosRecord = CognosRecord[['Real Customer ID 1', 'CIBM/COB/BUB team']]
#     CAListMst = CAListMst.merge(CognosRecord, left_on='Customer_ID_S1', right_on='Real Customer ID 1', how='left')
# except:
#     CognosRecord = pd.DataFrame()
# print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])
####### These should be commented - End #######

CAListMst = CAListMst[np.logical_not(CAListMst['Customer_ID_S1'].isnull())]
CMAll = pd.read_excel(CMFullList_File, 'CM list')
CMFullList = pd.read_excel(FTEFile, 'Sheet1')
CMFullList['CMTH'] = np.char.upper(CMFullList['CMTH'].tolist())

WIPFile = getFormatFile('Assigned MG without T0', '_', '.xlsx', 1)
NW_Ass_Case = pd.read_excel(WIPFile, 'Non WIP Assigned Case')



####### Mapping the Latest CM Team by using CM e-mail name in for WIP
WIPMI['Master-CM Team Head Old'] = WIPMI['Master-CM Team Head']
WIPMI = WIPMI.drop('Master-CM Team Head', axis=1)
CMAll['TH email'] = CMAll['TH email'].apply(lambda x: ModCMTH(x))
CMAll['Case Manager Email'] = CMAll['Case Manager Email'].apply(lambda x: ModCMTH(x))
WIPMI['Master-CM Team Head'] = WIPMI['Master-CM Name'].apply(lambda x: MapCMTeam(x, CMAll))
WIPMI.loc[WIPMI['Master-CM Team Head'] == 'Not Match', 'Master-CM Team Head'] = WIPMI[WIPMI['Master-CM Team Head'] == 'Not Match']['Master-CM Team Head Old']
WIPMI = WIPMI.drop('Master-CM Team Head Old', axis=1)


####### Read MGCSM Handling Cases
CMTeam = CMFullList.copy(deep=True)
CMTeam = CMTeam.set_index('CMTH')
MS90CMTeam = pd.read_excel(MasterGp_List)
try:
    MS90CMTeam = MS90CMTeam[MS90CMTeam['Remarks'] != 0]
except:
    pass
if 'MG CM owner' in MS90CMTeam.columns.tolist():
    MS90CMTeam['MG CM Name'] = MS90CMTeam['MG CM owner'].apply(lambda x: GetMGCM_Name(x))
    #print(MS90CMTeam.head(10))
    MS90CMTeam = MS90CMTeam.rename(columns={'MG CM owner': 'MG CM E-Mail Name'})
    MS90CMTeam = MS90CMTeam.rename(columns={'MG CM Name': 'MG CM'})
else:
    MS90CMTeam['MG CM Name'] = MS90CMTeam['MG CM'].apply(lambda x: GetMGCM_Name(x))
    MS90CMTeam = MS90CMTeam.rename(columns={'MG CM': 'MG CM E-Mail Name'})
    MS90CMTeam = MS90CMTeam.rename(columns={'MG CM Name': 'MG CM'})


####### Read Forced Assignment List

ForcedCase = pd.read_excel(ForcedCaseAssigment_File)



####### Merging Master Group info (horis) to Case Assignmen File
# ColCAListMst = CAListMst.columns.tolist()
CAListMst = pd.merge(CAListMst, MGFL, how='left', left_on='CIN', right_on='CIN', suffixes=(False, False))
CAListMst = CAListMst.drop(['Customer_Name', 'Master Group ID', 'Master Group'], axis=1)
# CAListMst = CAListMst.drop(['CIN', 'Customer_Name', 'Master Group ID', 'Master Group'], axis=1)

tmpCols = CAListMst.columns.tolist()
for i in range(0, len(tmpCols)):
    if tmpCols[i] == 'Master_Group_ID':
        tmpCols[i] = 'Master Group ID'
    elif tmpCols[i] == 'Master_Group_Name':
        tmpCols[i] = 'Master Group'

        
CAListMst.columns = tmpCols

Sel = CAListMst['CMB_Segment_S1'] == 'BBRM'
CAListMst.loc[Sel, 'Master Group ID'] = 0
CAListMst.loc[Sel, 'Master Group'] = ''

####### Randomize the sequence of Case Assignment List
CAListMst['Org_Seq'] = CAListMst.index.tolist()
CAListMst['Asignment Seq'] = indexMapper(size=CAListMst.shape[0])
#CAListMst.to_excel('pre-assign.xlsx')
CAListMst = CAListMst.sort_values(by='Asignment Seq')

print(np.sum(CAListMst['Master Group ID'].isnull()))

print('Total No of case to be assigned:\t%s' % str(CAListMst.shape[0]))

File found:	Assigned MG without T0_20260817100807.xlsx
0
Total No of case to be assigned:	4


In [38]:
####### Update KYC Opd Teams into SkippedTeam list

Sel = CMFullList['Segment'] == 'KYC Ops'
KYCOps_CMTH = list(CMFullList.loc[Sel, 'CMTH'])
if len(KYCOps_CMTH) > 0:
    SkippedTeam = list(dict.fromkeys(SkippedTeam + KYCOps_CMTH))
print('Updated Skipped Team:')
print(SkippedTeam)

Updated Skipped Team:
['MARCO Y O CHUNG', 'Vasuda VENKATADRI', 'Ken Y K FONG', 'Cherry H Y MAK']


In [39]:
####### Merging IMIS Group info to Case Assignmen File
IMIS_No = []
IMIS_GName = []
CAListMst = CAListMst.set_index('Customer_ID_S1', drop=False)
for index, row in CAListMst.iterrows():
    SelInd = index == IMIS['Real customer id']
    #print('Find:\t%d'% np.sum(SelInd))
    if np.sum(SelInd) > 0:
        #print(list(IMIS[SelInd]['IMIS Group No'])[0])
        if pd.isnull(list(IMIS[SelInd]['IMIS Group No'])[0]):
            #print('\tIs Null')
            IMIS_No.append('')
        else:
            #print('\tNot Null')
            IMIS_No.append(list(IMIS[SelInd]['IMIS Group No'])[0])
        IMIS_GName.append(list(IMIS[SelInd]['Group Name'])[0])
    else:
        
        IMIS_No.append('')
        IMIS_GName.append('')
        
CAListMst['IMIS Group No'] = IMIS_No
CAListMst['IMIS Group'] = IMIS_GName
Sel = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')

CAListMst.loc[Sel, ['IMIS Group No', 'IMIS Group']] = ''
#CAListMst.set_index('Org_Seq', drop=True)
#print(len(IMIS_No))
#print(len(IMIS_GName))
#print(CAListMst.shape[0])

In [40]:
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


# Prepare for Case assignment

In [41]:
skipped = SkipTeam(SkippedTeam, CMFullList)
#skipped = ''
if skipped != '':
    print('Team to be skipped case assignment:')
    print(skipped)
else:
    print('No skipped Team')

Team to be skipped case assignment:
['KEN Y K FONG']


In [42]:
####### To check if tehre is any case to be assiged to ETB Team from MGCSM Team via Forced Assignment List or vice versa


nMG2ETB_real = 0
nMG2ETB_GP_real = 0
if np.sum(ForcedCase['CMTH'] == 'ETB CM') > 0:
    print('There is MSCSM case transfered to ETB Team')
    CheckMGID = CAListMst['Master Group ID'][np.logical_and(CAListMst['Master Group ID'] > 0, CAListMst['CMB_Segment_S1'] == 'CORP')].tolist()
    ForcedMG2ETBList = ForcedCase['Master Group ID'][ForcedCase['CMTH'] == 'ETB CM'].tolist()
    
    for aMGID in CheckMGID:
        #print(aMGID)
        if aMGID in ForcedMG2ETBList:
            nMG2ETB_GP_real += 1
            
    CheckMGCusID = CAListMst['Customer_ID_S1']
    SelFMGID01 = (ForcedCase['Real ID'] != '')
    SelFMGID02 = (ForcedCase['CIN'] > 0)
    SelFMGID03 = (ForcedCase['CMTH'] == 'ETB CM')
    SelFMGID = np.logical_or(SelFMGID01, SelFMGID02)
    SelFMGID = np.logical_and(SelFMGID, SelFMGID03)
    ForcedCusID = ForcedCase['Real ID'][SelFMGID]
    nMG2ETB_real += np.sum(CheckMGCusID.isin(ForcedCusID))
    print('The number of case assigned back to ETB:\t%d' % nMG2ETB_real)
    print('The number of Group assigned back to ETB:\t%d' % nMG2ETB_GP_real)

In [43]:
# Use tuple to get the record

# Table for showing no of case in RM Team for this batch of case assignment
# RMTeamCount = pd.pivot_table(CAListMst, index=['CMB_Segment_S1', 'RM_Team_Group_S1'], values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

# Table for showing no of CM in RM Team for this batch of case assignment
#CMTeam = pd.pivot_table(CMFullList, index=['TH email'], values=['Case Manager Name'] , aggfunc='count', fill_value=0)


if skipped == '':
    CMTeam['Skipped'] = False
else:
    CMTeam['Skipped'] = CMTeam.index.isin(skipped)
    
####### Update Capacity for Case Assignment, excluding Split Team
####### Actual Capacity for Case Assignment will be further updated later

TotalCM = np.sum(CMTeam[np.logical_not(CMTeam['Skipped'])]['FTE'])
print('Total Number of CM (FTE):\t%d' % TotalCM)
CMTeam['% HP'] = np.logical_not(CMTeam['Skipped']) * CMTeam['FTE'] / TotalCM
#CMTeam['No of BB Case'] = np.ceil(CMTeam['% HP'] * CAListMst[CAListMst['CMB_Segment_S1'] == 'BBRM'].shape[0])
#CMTeam['No of Corp Case'] = np.ceil(CMTeam['% HP'] * CAListMst[CAListMst['CMB_Segment_S1'] == 'CORP'].shape[0])

MGList = MS90CMTeam['Master Group ID'].tolist()
CAListMst.loc[:, 'Master Group ID'] = CAListMst['Master Group ID'].fillna('')

#*********************************************************************************************#
if  nMG2ETB_real == 0:
    nMG2ETB_real = np.sum(CAListMst['Master Group ID'].isin(MGList))


nETB = (CAListMst.shape[0]) - nMG2ETB_real
#print(nETB)
print('Total Number of Case for ETB Team:\t%d' % nETB)
print('Total Number of Case for MG Team:\t%d' % nMG2ETB_real)
CMTeam['No of Total Case'] = np.ceil(CMTeam['% HP'] * nETB)

CMTeam['Assign Seq'] = indexMapper(CMTeam.shape[0])

#CMTeam['Assign Seq'] = list(range(0, CMTeam.shape[0]))

CMTeam['Total No of BBRM Assigned'] = 0
CMTeam['Total No of Corp Assigned'] = 0
CMTeam['Total No of CMB Assigned'] = 0
CMTeam['Total No of Assigned'] = 0

######## Needs to be updated #######
CMTeam['Capacity'] = np.ceil(CMTeam['% HP'] * CAListMst.shape[0])
CMTeam = ModCMTHIndex(CMTeam)

CMTeam = CMTeam.sort_values(by='Assign Seq')
try:
    CAListMst['WIP MG'] = np.logical_and(CAListMst['Master Group ID'].isin(WIPMI['MasterGroup ID']), CAListMst['Master Group ID'] != '')
except:
    try:
        CAListMst['WIP MG'] = np.logical_and(CAListMst['Master Group ID'].isin(WIPMI['MasterGroup ID']), CAListMst['Master Group ID'].replace(0, '') != '')
    except:
        CAListMst['WIP MG'] = CAListMst['Master Group ID'].isin(WIPMI['MasterGroup ID'])
CAListMst['WIP IMIS ID'] = np.logical_and(CAListMst['IMIS Group No'].isin(WIPMI['IMISGroupNo']), CAListMst['IMIS Group No'] != '')



print(np.sum(CAListMst['WIP MG']))
print(np.sum(CAListMst['WIP IMIS ID']))

CAListMst['Case_Manager_Email_S1'] = ''
CAListMst['Case_Manager_Staff_ID_S1'] = ''
CAListMst['Case_Manager_Telephone_S1'] = ''
CAListMst['Assign by TH'] = ''
CAListMst = CAListMst.fillna('')
CAListMst['Backup Team Head'] = ''

print(CAListMst['CMB_Segment_S1'].value_counts())

Total Number of CM (FTE):	60
Total Number of Case for ETB Team:	4
Total Number of Case for MG Team:	0
4
0
CORP    4
Name: CMB_Segment_S1, dtype: int64


In [44]:
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


# Basic Statistic

In [45]:
####### For Reference only, on the workload of the Team
####### Not Affect upcoming 

# WorkLoadTH = pd.pivot_table(WIPMI, index=['Master-CM Team Head'], values=['CIN'] , aggfunc='count', fill_value=0)
# nCMPT = pd.pivot_table(WIPMI, index=['Master-CM Team Head'], values=['Master-CM Name'] , aggfunc='count', fill_value=0)

# WIP_Stat = pd.DataFrame(columns=['CM Team', 'CM Name', 'WIP Case'])
# WIP_Stat_Team = pd.DataFrame(columns=['CM Team', 'No of CM Name', 'WIP Case', 'Case per CM'])
# WIPCMTH = WIPMI['Master-CM Team Head'].dropna().unique().tolist()
# i = 0
# j = 0
# k = 0
# for aTH in WIPCMTH:
#     CMList = WIPMI[WIPMI['Master-CM Team Head'] == aTH]['Master-CM Name'].unique().tolist()
    
#     j = j + 1
#     k = 0
#     for aCM in CMList:
#         WIP_Stat.loc[i] = [aTH, aCM, 0]
#         i = i + 1
#         k += 1
#         pass
#     WIP_Stat_Team.loc[j] = [aTH, k, 0, 0]
#     pass

# for aCM in WIP_Stat['CM Name'].tolist():
#     nC = np.sum(WIPMI['Master-CM Name'] == aCM)
#     WIP_Stat.loc[WIP_Stat['CM Name'] == aCM, 'WIP Case'] = nC
#     pass

# for aTH in WIPCMTH:
    
#     nC = np.sum(WIP_Stat[WIP_Stat['CM Team'] == aTH]['WIP Case'])
#     WIP_Stat_Team.loc[WIP_Stat_Team['CM Team'] == aTH, 'WIP Case'] = nC
    
# #WIP_Stat_Team['Case per CM'] = WIP_Stat_Team['WIP Case'] / WIP_Stat_Team['No of CM Name']
# for aCM in WIP_Stat_Team['CM Team'].tolist():
#     if aCM != 'MARCO Y O CHUNG' and aCM != 'Not Match' and aCM in CMTeam.index.tolist():
#     #if aCM != 'Not Match' and aCM in CMTeam.index.tolist():
#         WIP_Stat_Team.loc[WIP_Stat_Team['CM Team'] == aCM, 'No of CM Name'] = CMTeam.loc[aCM]['FTE']

# WIP_Stat_Team['Case per CM'] = WIP_Stat_Team['WIP Case'] / WIP_Stat_Team['No of CM Name']
# WIP_Stat_Team['Case per CM'] = 0
# Avg_All = np.sum(WIP_Stat_Team['WIP Case']) / np.sum(WIP_Stat_Team['No of CM Name'])
# WIP_Stat_Team['Diff from Overall Average'] = WIP_Stat_Team['Case per CM'] - Avg_All
# WIP_Stat_Team

In [46]:
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [47]:
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


In [48]:
#SLoc = 0
#THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)

In [49]:
#THSeqForAssg(CMTeam, 2, 'Capacity', False, SLoc)

# Read File for Corp Cases handled by KYC Ops

In [50]:
if KYC_CM_Exist:
    KYCOps_Corp = pd.read_excel(KYCOps_Corp_File)

# Map MGCSM CM to Column Selected MGs

In [51]:
# MGIDExtract = CAListMst[['Customer_ID_S1', 'CMB_Segment_S1', 'Master Group ID']]

In [52]:
# MGIDExtract_ = MGIDExtract.copy(deep=True)

# print(MGIDExtract_)

# MGIDExtract_['Master Group ID'] = MGIDExtract_['Master Group ID'].apply(lambda x: 0 if x== '' else x)
# MGIDExtract_2 = MGIDExtract_.merge(MS90CMTeam[['Master Group ID', 'MG CM']], how='left', left_on='Master Group ID', right_on='Master Group ID')
# print(MGIDExtract_)

# MGIDExtract_2['Master Group ID'] = MGIDExtract_2['Master Group ID'].apply(lambda x: '' if x == 0 else x)
# MGIDExtract_2['MG CM'] = MGIDExtract_2['MG CM'].fillna('')

# MGIDExtract_2

In [53]:
MGIDExtract = CAListMst[['Customer_ID_S1', 'CMB_Segment_S1', 'Master Group ID']]
#1461-MGIDExtract
if 'MG CM' in MS90CMTeam.columns.tolist():
    print('Route A')
    
    try:
        
        MGIDExtract = MGIDExtract.merge(MS90CMTeam[['Master Group ID', 'MG CM']], how='left', left_on='Master Group ID', right_on='Master Group ID')
    except ValueError:
        
        MGIDExtract['Master Group ID'] = MGIDExtract['Master Group ID'].apply(lambda x: 0 if x== '' else x)
        #print(MGIDExtract)
        print('--------------------------')
        MGIDExtract = MGIDExtract.merge(MS90CMTeam[['Master Group ID', 'MG CM']], how='left', left_on='Master Group ID', right_on='Master Group ID')
        #MGIDExtract = MGIDExtract.merge(MS90CMTeam[['Master Group ID', 'MG CM']], how='left', on='Master Group ID')
        print(MGIDExtract)
        #1467-MGIDExtract
        #MGIDExtract.to_csv (r'C:\Users\44009045\AppData\Roaming\Python\Python39\Scripts\CA1216BOW\export_dataframe.csv', index = None, header=True)
        MGIDExtract['Master Group ID'] = MGIDExtract['Master Group ID'].apply(lambda x: '' if x == 0 else x)
        MGIDExtract['MG CM'] = MGIDExtract['MG CM'].fillna('')
    
    MGIDExtract['MG CM'] = MGIDExtract['MG CM'].fillna('')
    Sel = MGIDExtract['CMB_Segment_S1'] == 'BBRM'
    MGIDExtract.loc[Sel, 'MG CM'] = ''
    MGIDExtract.loc[Sel, 'Master Group ID'] = ''
    #print(MGIDExtract)
    CAListMst['Selected MGs'] = MGIDExtract['MG CM'].values
    pass
else:
    print('Route B')
    MGIDExtract = MGIDExtract.merge(MS90CMTeam[['Master Group ID', 'MG CM owner']], how='left', left_on='Master Group ID', right_on='Master Group ID')
    MGIDExtract['MG CM owner'] = MGIDExtract['MG CM owner'].fillna('')
    Sel = MGIDExtract['CMB_Segment_S1'] == 'BBRM'
    MGIDExtract.loc[Sel, 'MG CM owner'] = ''
    MGIDExtract.loc[Sel, 'Master Group ID'] = ''
    CAListMst['Selected MGs'] = MGIDExtract['MG CM owner'].values
    pass
CAListMst['Selected MGs'] = CAListMst['Selected MGs'].str.title()

Route A


In [54]:
np.sum(CAListMst['Selected MGs'] != '')

0

In [55]:
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


# Map MGCSM CM Staff ID to Column MG CM ID

In [56]:
CMAll[['Case Manager Name', 'Case Manager Staff ID']].head()

,Case Manager Name,Case Manager Staff ID
0,Caiden Lau,43819301
1,Cri Wong,43861481
2,Cyrus Wong,43899854
3,Eric Chung,43644022
4,Eunice Leung,43399034


In [57]:
MGCMID = CAListMst[['Selected MGs']]
MGCMID = MGCMID.merge(CMAll[['Case Manager Name', 'Case Manager Staff ID']], how='left', left_on='Selected MGs', right_on='Case Manager Name')
Sel_MultiCM = MGCMID['Selected MGs'].str.contains('/')

if np.sum(Sel_MultiCM) > 0:

    FullCM = '/'.join(MGCMID['Selected MGs'][Sel_MultiCM].unique().tolist()).split('/')
    CM_Dict = {}
    for aCM in FullCM:
        aCM_s = GetMGCM_Name(aCM)

        SelCM = CMAll['Case Manager Name'] == aCM_s
        CMID = CMAll['Case Manager Staff ID'][SelCM].values[0]

        CM_Dict[aCM] = CMID


    for aCM in MGCMID['Selected MGs'][Sel_MultiCM].unique().tolist():
        CMID = ['']
        for aCM_s in aCM.split('/'):
            CMID = CMID + [str(CM_Dict[aCM_s])] + ['/']
        SelCM = MGCMID['Selected MGs'] == aCM
        MGCMID.loc[SelCM, 'Case Manager Staff ID'] = ''.join(CMID)[:-1]
    
MGCMID = MGCMID.fillna('')
CAListMst['MG CM ID'] = MGCMID['Case Manager Staff ID'].tolist()

In [58]:
# MGCMID = CAListMst[['Selected MGs']]
# MGCMID = MGCMID.merge(CMAll[['Case Manager Name', 'Case Manager Staff ID']], how='left', left_on='Selected MGs', right_on='Case Manager Name')
# MGCMID = MGCMID.fillna('')
# CAListMst['MG CM ID'] = MGCMID['Case Manager Staff ID'].tolist()
# #CAListMst['MG CM ID']

In [59]:
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


# Map BRM Info as Boolean

In [60]:
if KYC_CM_Exist:
#     BRM_CustID = BRM['Customer ID']
#     CAListMst['BRM Case'] = CAListMst['Customer_ID_S1'].isin(BRM_CustID)
    BRM_StaffID = BRM['RM ID (AFTER)']
    CAListMst['BRM Case'] = CAListMst['RM_Staff ID_S1'].isin(BRM_StaffID)

# Map RM Sub Team for Corp Case

In [61]:
if KYC_CM_Exist:
    RMSubTeam = RMSubTeam.drop_duplicates(subset=['REAL_CUST_ID_1'])

In [62]:
if KYC_CM_Exist:
    Sel = RMSubTeam['REAL_CUST_ID_1'].isin(CAListMst['Customer_ID_S1'])
    RMSubTeam = RMSubTeam[Sel]
    RMSubTeam = RMSubTeam[['REAL_CUST_ID_1', 'TEAM', 'SUB_TEAM']] 
    Sel = RMSubTeam['SUB_TEAM'].isnull()
    RMSubTeam.loc[Sel, 'SUB_TEAM'] = RMSubTeam['TEAM'][Sel]
    tmpSubTeam = CAListMst['RM Sub Team']
    
    try:
        CAListMst = CAListMst.merge(RMSubTeam[['REAL_CUST_ID_1', 'SUB_TEAM']], how='left', left_on='Customer_ID_S1', right_on='REAL_CUST_ID_1')
    except ValueError:
        CAListMst = CAListMst.reset_index(drop=True)
        CAListMst = CAListMst.merge(RMSubTeam[['REAL_CUST_ID_1', 'SUB_TEAM']], how='left', left_on='Customer_ID_S1', right_on='REAL_CUST_ID_1')
        CAListMst = CAListMst.set_index('Customer_ID_S1')
    
    CAListMst['RM Sub Team'] = CAListMst['SUB_TEAM']
    CAListMst = CAListMst.drop(['REAL_CUST_ID_1', 'SUB_TEAM'], axis=1)
    
    Sel = ((CAListMst['RM Sub Team'] == '') | pd.isnull(CAListMst['RM Sub Team'])) & (CAListMst['CMB_Segment_S1'] == 'CORP') & \
    (pd.notnull(CAListMst['CIBM/COB/BUB team']) | (CAListMst['CIBM/COB/BUB team'].str.strip() != ''))
    
    if np.sum(Sel) > 0:
        print('No of Corp Case without Sub Team:\t%d' % np.sum(Sel))
        
        for aTeamCode in CAListMst[Sel]['CIBM/COB/BUB team'].tolist():
            print(aTeamCode)
            SelCase = Sel & (CAListMst['CIBM/COB/BUB team'] == aTeamCode)
            SelSubTeam = RMSubTeamCorp_Manual['RM Team Group'] == aTeamCode
            print(np.sum(SelSubTeam))
            if np.sum(SelSubTeam) > 0:
                CAListMst.loc[SelCase, 'RM Sub Team'] = RMSubTeamCorp_Manual[SelSubTeam]['Sub Team'].values[0]
            else:
                print(aTeamCode)
        
        


In [63]:
####### If Sub team cannot be located in Master MI, will refer to the RM Record and check with Master MI

if KYC_CM_Exist:
    WIPMI_C = WIPSelectot(WIPMI_C, WIP_Day)
    Sel01 = pd.isnull(CAListMst['RM Sub Team'])
    RMList = CAListMst[Sel01]['RM_Name_S1'].unique()
    for aRM in RMList:
        Sel02 = CAListMst['RM_Name_S1'] == aRM
        Sel = np.logical_and(Sel01, Sel02)
        if np.sum(Sel) > 0:
            SelMI_Team = WIPMI_C['Team'] == CAListMst[Sel]['RM_Team_Group_S1'].tolist()[0]
            
            
            if np.sum(SelMI_Team) > 0:
                print(aRM)
                WIPMI_Sec = WIPMI_C.loc[SelMI_Team, ['RM Name', 'Team', 'RM Sub Team']]
                CAListMst.loc[Sel, 'RM Sub Team'] = WIPMI_Sec['RM Sub Team'].value_counts().idxmax()
                print('Locate Sub Team')
            else:
                print(aRM)
                expString = '|'.join(str.split(aRM))
                #print(expString)
                SelMI_RM = WIPMI_C['RM Name'].str.contains(aRM, na=False)
                SelMI_Team = np.logical_and(SelMI_Team, SelMI_RM)
                if np.sum(SelMI_Team) > 0:
                    #print(WIPMI_Sec['RM Sub Team'].value_counts().idxmax())
                    WIPMI_Sec = WIPMI_C.loc[SelMI_Team, ['RM Name', 'Team', 'RM Sub Team']]
                    CAListMst.loc[Sel, 'RM Sub Team'] = WIPMI_Sec['RM Sub Team'].value_counts().idxmax()
                    print('Locate Sub Team')
                else:
                    print('Cannot Locate Sub Team')
        else:
            print(aRM)
            print('Cannot Locate Sub Team')
                

In [64]:
np.sum(WIPMI_C['RM Name'].str.contains('CHAN KING MAN MEMPHIS'))

0

In [65]:
####### If Sub team cannot be located in Master MI, will refer to the value in  CAListMst file

if KYC_CM_Exist:
    try:
        Sel01 = pd.isnull(CAListMst['RM Sub Team'])
        CustList = CAListMst[Sel01]['Customer_ID_S1'].tolist()
    except KeyError:
        CAListMst = CAListMst.reset_index(drop=False)
        Sel01 = pd.isnull(CAListMst['RM Sub Team'])
        CustList = CAListMst[CAListMst == pd.isnull(CAListMst['RM Sub Team'])]['Customer_ID_S1'].tolist()
    
    for aCust in CustList:
        Sel02 = CAListMst['Customer_ID_S1'] == aCust
        
        Sel = np.logical_and(Sel01, Sel02)
        if np.sum(Sel) > 0:
            CAListMst.loc[Sel, 'RM Sub Team'] = tmpSubTeam.loc[aCust]

In [66]:
print('********\tCAListMst Size:\t% d' % CAListMst.shape[0])

********	CAListMst Size:	 4


# KC Comment's
    
    1. Assign case for per Forced Assignment List (ignore skipping)
    2. Assign case for MGCSM (ignore skipping)
    3. Assign case for MG Group client in previous case assignment but without T0 (Corp) (ignore skipping)
    4. Assign case for IMIS Group client in previous case assignment but without T0 (BBRM) (ignore skipping)
    5. Assign case for WIP MG Team (Corp) (ignore skipping)
    6. Assign case for WIP IMIS ID (BBRM) (ignore skipping)
    7. Assign case for CORP cases within the same Master Group (without previous assignment and non-WIP)
    8. Assign case for BBRM cases within the same IMIS Group (without previous assignment and non-WIP)
    9. Assign case without MG ID / IMIS Group
    10. Code to ensure no un-assigned case

# 1. Code for Forced Case Assignment

In [67]:
CINs = ForcedCase['CIN'].tolist()
MGIDs = ForcedCase['Master Group ID'].tolist()
CustID = ForcedCase['Real ID'].tolist()

BBExtist = 'IMIS ID' in ForcedCase.columns.tolist()
if BBExtist:
    RGIDs = ForcedCase['IMIS ID'].tolist()

cc_Number = 0
fasg_Num = 0
for i in range(0, ForcedCase.shape[0]):
    
    if (ForcedCase.loc[i]['Expiry Date'] >= pd.datetime.now()) or pd.isnull(ForcedCase.loc[i]['Expiry Date']):
        cc_Number += 1
        if pd.notnull(CINs[i]):
            Sel = CAListMst['Customer_CIN_S1'] == CINs[i]
            fasg_Num += np.sum(Sel)
        elif pd.notnull(CustID[i]):
            Sel = CAListMst['Customer_ID_S1'] == CustID[i]
            fasg_Num += np.sum(Sel)
        elif pd.notnull(MGIDs[i]):
            Sel = CAListMst['Master Group ID'] == MGIDs[i]
            SelCP = CAListMst['CMB_Segment_S1'] == 'CORP'
            Sel = np.logical_and(Sel, SelCP)
            fasg_Num += np.sum(Sel)
        else:
            if BBExtist:
                Sel = CAListMst['IMIS ID'] == RGIDs[i]
                SelCP = CAListMst['CMB_Segment_S1'] == 'BBRM'
                Sel = np.logical_and(Sel, SelCP)
                fasg_Num += np.sum(Sel)
                
            
        if np.sum(Sel) > 0:
            CAListMst.loc[Sel, 'Assign by TH'] = ForcedCase.loc[i]['CMTH']
            CAListMst.loc[Sel, 'CM Name'] = ForcedCase.loc[i]['CM Name']
            CAListMst.loc[Sel, 'Assigment Method'] = 'Forced Assignment'
            

print('No of active case with forced assignment:\t%d' % cc_Number)
print('No of forced case assignment in this batch:\t%d' % fasg_Num)
print('No of active case with forced assignment back to CM Team from MGCSM:\t%d' % np.sum(CAListMst['CM Name'] == 'ETB CM'))
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

########  Debug Code ########
#CAListMst.to_excel('CM Assg - 01 Force Assignment.xlsx')

No of active case with forced assignment:	0
No of forced case assignment in this batch:	0
No of active case with forced assignment back to CM Team from MGCSM:	0


C:\Users\44009045\AppData\Local\Temp\ipykernel_32152\4016699.py:13: FutureWarning: The pandas.datetime class is deprecated and will be removed from pandas in a future version. Import from datetime module instead.
  if (ForcedCase.loc[i]['Expiry Date'] >= pd.datetime.now()) or pd.isnull(ForcedCase.loc[i]['Expiry Date']):


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [68]:
####### Debug Checking #######
# CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

In [69]:
# 1.1 BRM case assignment 

# 2. Code for MGCSM Team

In [70]:
#### For RCSM (New) ####
CMAll_RCSM = CMAll[CMAll['TH email']==RCSM_TH.upper()]
CMAll_RCSM['Case Manager Email'] = CMAll_RCSM['Case Manager Email'].str.upper()

In [71]:
Start_MG90 = 1
MGCount = 0
if Start_MG90 :
    
    
    #pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)
    try:
        MGList = CAListMst['Master Group ID'][np.logical_and((CAListMst['Master Group ID'] != ''), CAListMst['CMB_Segment_S1'] == 'CORP')].tolist()
    except:
        try:
            MGList = CAListMst['Master Group ID'][np.logical_and((CAListMst['Master Group ID'].replace(0, '') != ''), CAListMst['CMB_Segment_S1'] == 'CORP')].tolist()
        except:
            MGList = CAListMst['Master Group ID'][np.logical_and((CAListMst['Master Group ID'] > 0), CAListMst['CMB_Segment_S1'] == 'CORP')].tolist()

    Sel01 = MS90CMTeam['Master Group ID'].isin(MGList)

    MGListCMT = MS90CMTeam['Master Group ID'][Sel01].tolist()
    nMGCSM = (np.sum(CAListMst['Master Group ID'][CAListMst['CMB_Segment_S1'] == 'CORP'].isin(MS90CMTeam['Master Group ID'].tolist())) - np.sum(CAListMst['CM Name'] == 'ETB CM'))
    
    print('Corp Case to be handled by MG CM Team:\t%d' % (np.sum(CAListMst['Master Group ID'][CAListMst['CMB_Segment_S1'] == 'CORP'].isin(MS90CMTeam['Master Group ID'].tolist())) - np.sum(CAListMst['CM Name'] == 'ETB CM')))
    print('BBRM Case to be handled by MG CM Team:\t%d' % np.sum(CAListMst['Master Group ID'][CAListMst['CMB_Segment_S1'] == 'BBRM'].isin(MS90CMTeam['Master Group ID'].tolist())))
    
    if np.sum(CAListMst['Master Group ID'].isin(MS90CMTeam['Master Group ID'].tolist())) > 0:
        for aMG in MGListCMT:
            SelMG = CAListMst['Master Group ID'] == aMG
            Sel01 = CAListMst['Assign by TH'] == ''
            Sel02 = CAListMst['CMB_Segment_S1'] == 'CORP'
            
            SelMG = np.logical_and(Sel01, SelMG)
            #find a masterorup has been assigned
            SelMG = np.logical_and(Sel02, SelMG)
        
            
            
            SelMG_c = MS90CMTeam['Master Group ID'] == aMG
            
            if 'MG CM' in MS90CMTeam.columns.tolist():
                MGCM_Name = MS90CMTeam[SelMG_c]['MG CM'].values[0]
                CAListMst.loc[SelMG, 'CM Name'] = MGCM_Name
                #CAListMst.loc[SelMG, 'Selected MGs'] = MS90CMTeam[SelMG_c]['MG CM'].values[0]
            else:
                MGCM_Name = MS90CMTeam[SelMG_c]['MG CM owner'].values[0]
                CAListMst.loc[SelMG, 'CM Name'] = MGCM_Name
                #CAListMst.loc[SelMG, 'Selected MGs'] = MS90CMTeam[SelMG_c]['MG CM owner'].values[0]
                
#             if MGCM_Name.upper() in CMAll_RCSM['Case Manager Email'].tolist():
                
            xxx = []
            for i in CMAll_RCSM['Case Manager Email'].tolist():
                i = i.split()
                xxx.append(i[0]+" "+i[-1])
                
                
            if MGCM_Name.upper() in xxx:   
                CAListMst.loc[SelMG, 'Assign by TH'] = RCSM_TH
                CAListMst.loc[SelMG, 'Assigment Method'] = 'RCSM'
            else:
                CAListMst.loc[SelMG, 'Assign by TH'] = MGCSM_TH
                CAListMst.loc[SelMG, 'Assigment Method'] = 'MGCSM'
            pass
            Start_MG90 += np.sum(SelMG)
            #print(CAListMst.loc[SelMG, ['Master Group', 'Selected MGs']])
        pass
    pass
# CAListMst.loc[CAListMst['Assign by TH'] == 'ETB CM', 'Assigment Method'] = ''
# CAListMst['Assign by TH'] = CAListMst['Assign by TH'].replace('ETB CM', '')
# CAListMst['CM Name'] = CAListMst['CM Name'].replace('ETB CM', '')
CMTeam

########  Debug Code ########
#CAListMst.to_excel('CM Assg - 02 MGCSM.xlsx')

Corp Case to be handled by MG CM Team:	0
BBRM Case to be handled by MG CM Team:	0


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [72]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
,4


In [73]:
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [74]:
####### Debug Checking #######
# CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 2.1 Code for Case Assignment (KYC Ops Handled BBRM Case)

In [75]:
####### To List the Cases Under KYC Ops #######
if KYC_CM_Exist:
    SelNoTH = CAListMst['Assign by TH'] == ''
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel_BRM = CAListMst['BRM Case']
    
    SelBBRM = np.logical_and(SelBBRM, Sel_BRM)
    
    KYCOps_Corp = KYCOps_Corp.fillna('')
    SelCorp = CAListMst['CMB_Segment_S1'] == 'CORP'
    SelCorp_KYCOps = np.repeat(False, CAListMst.shape[0])
    for index, rows in KYCOps_Corp.iterrows():
        SelTeam = CAListMst['RM_Team_Group_S1'] == rows['Team']
        if rows['Sub Team'] ==  '':
            SelSubTeam = CAListMst['RM_Name_S1'] == rows['RM Name']
        else:
            SelSubTeam = CAListMst['RM Sub Team'] == rows['Sub Team']
        Sel = np.logical_and(SelTeam, SelSubTeam)
        if rows['Sub Team'] ==  '':
            print('%s : %s:\t%d' % (rows['Team'], rows['RM Name'], np.sum(Sel)))
        else:
            print('%s : %s:\t%d' % (rows['Team'], rows['Sub Team'], np.sum(Sel)))
        SelCorp_KYCOps = np.logical_or(Sel, SelCorp_KYCOps)
    
    
    SelCorp_KYCOps = np.logical_and(SelCorp, SelCorp_KYCOps)
    
    SelKYCOpsCase = np.logical_or(SelCorp_KYCOps, SelBBRM)
    SelKYCOpsCase = np.logical_and(SelKYCOpsCase, SelNoTH)

    CAListMst['KYC Ops Corp Case'] = SelCorp_KYCOps
    
    
    
    CAListMst['Case Handled by KYC Ops'] = SelKYCOpsCase
    

In [76]:
if KYC_CM_Exist:
    print('Total Number of case for KYC Ops CM:')
    print(np.sum(CAListMst['Case Handled by KYC Ops']))

In [77]:
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [78]:
if KYC_CM_Exist:
    KYCOpsCMTeam = CMTeam[CMTeam['Segment'] == 'KYC Ops']

    KYCOpsCMTeam['Skipped'] = False

    KYCOpsCMTeam['% HP'] = KYCOpsCMTeam['FTE'] / np.sum(KYCOpsCMTeam['FTE'])

    KYCOpsCMTeam['No of Total Case'] = np.ceil(KYCOpsCMTeam['% HP'] * np.sum(CAListMst['Case Handled by KYC Ops']))
    
    KYCOpsCMTeam['Assign Seq'] = indexMapper(KYCOpsCMTeam.shape[0])
    KYCOpsCMTeam = KYCOpsCMTeam.sort_values(by='Assign Seq')
    
    KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)
    print(KYCOpsCMTeam)


In [79]:
####### To show number of BBRM cases under KYC Ops #######

if KYC_CM_Exist:
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel_BRM = CAListMst['BRM Case']
    SelBBRM_KYCOps = np.logical_and(SelBBRM, Sel_BRM)
    print('No of BBRM cases handled by KYC Ops CM Team:\t%d' % np.sum(SelBBRM_KYCOps))

In [80]:
####### Code for case with WIP IMIS Group #######
####### Checked, Okay #######

if KYC_CM_Exist and np.sum(SelBBRM_KYCOps) > 0:
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel_BRM = CAListMst['BRM Case']
    SelBBRM_KYCOps = np.logical_and(SelBBRM, Sel_BRM)

    CorpSel00 = CAListMst['Assign by TH'] == ''
    CorpSel01 = CAListMst['IMIS Group No'] != ''
    CorpSel02 = CAListMst['WIP IMIS ID']
    
    #print(CAListMst[CAListMst['IMIS Group No'] == 2620]['WIP IMIS ID'])
    CorpSel = np.logical_and(CorpSel00, CorpSel02)
    CorpSel = np.logical_and(CorpSel, CorpSel01)
    CorpSel = np.logical_and(CorpSel, SelBBRM_KYCOps)
    print('No of case to be assigned:\t%d' % np.sum(CorpSel))
    ttl_IMIS_WIP = 0
    if np.sum(CorpSel) > 0:
        WIPRID_Ext = WIPMI.copy(deep = True)
        #WIPRID_Ext = WIPRID_Ext[np.logical_not(WIPRID_Ext['IMISGroupNo'].isnull())]
        
        #WIPRID_Ext = WIPRID_Ext[(WIPRID_Ext['IMISGroupNo'] != '')]
        WIPRID_Ext = WIPRID_Ext.sort_values(by='T0', ascending=False)
        WIPRID_Ext = WIPRID_Ext.drop_duplicates(['IMISGroupNo'], keep='first')
#         print(WIPRID_Ext[WIPRID_Ext['IMISGroupNo'] == 2620][['Customer ID', 'Master-CM Name', 'Master-CM Team Head', 'T0']])
        CorpMG = CAListMst[CorpSel]
        #print(CorpMG.columns.tolist())
        CorpMG = pd.merge(CorpMG, WIPRID_Ext[['IMISGroupNo', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='IMIS Group No', right_on='IMISGroupNo', suffixes=(False, False))
        #print(CorpMG.shape)
        CorpMG = CorpMG[np.logical_not(CorpMG['T0'].isnull())][['IMIS Group No', 'Master-CM Team Head', 'Master-CM Name']]
        #print(CorpMG.shape)
        #CAListMst.loc[CorpSel, ['Assign by TH']] = CorpMG['Master-CM Team Head'].tolist()

        CorpMG = pd.merge(CorpMG, CMTeam[['Skipped']], how='left', left_on='Master-CM Team Head', right_index=True, suffixes=(False, False))
        #CorpMG = CorpMG[np.logical_not(CorpMG['Skipped'])]
        #print(CorpMG.columns)
        #print(CorpMG['IMIS Group No'] == )
        #print(CorpMG.index.tolist())
        for aMGIND in CorpMG.index.tolist():
            #print('IMIS ID:\t%d' % aMGIND)
            SelMG = CAListMst['IMIS Group No'] == CorpMG.loc[aMGIND]['IMIS Group No']
            #if aMGIND == 47041:
            #    print(np.sum(SelMG))
            ttl_IMIS_WIP += np.sum(SelMG)
            if np.sum(SelMG) > 0:
                #print('Team:\t%s\tCM:\t%s' % (CorpMG.loc[aMGIND]['Master-CM Team Head'], CorpMG.loc[aMGIND]['Master-CM Name']))
                print(CorpMG.loc[aMGIND]['Master-CM Team Head'])
                #print(KYCOps_CMTH)
                if not CorpMG.loc[aMGIND]['Master-CM Team Head'] in KYCOps_CMTH:
#                 if np.sum(CorpMG.loc[aMGIND]['Master-CM Team Head'] != KYCOps_CMTH):
                    print('Re-Assigned.')
                    print(THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)[0])
                    CAListMst.loc[SelMG, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
                    #CAListMst.loc[SelMG, 'Assign by TH'] = KYCOps_CMTH
                    #CAListMst.loc[SelMG, 'CM Name'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'Assigment Method'] = 'WIP (BBRM) (KYC Ops) (Tsf WIP to KYC Ops)'
                    if np.sum(SelMG) > 1:
                        CAListMst.loc[SelMG, 'CM Name'] = 'CM' + str(iCM)
                        iCM += 1
                    
                else:
                    print('Assigned.')
                    CAListMst.loc[SelMG, 'Assign by TH'] = CorpMG.loc[aMGIND]['Master-CM Team Head']
                    CAListMst.loc[SelMG, 'CM Name'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'Assigment Method'] = 'WIP (BBRM) (KYC Ops)'
                    
                #print('-------------------------------------')
                #print(CAListMst.loc[SelMG, 'Assign by TH'])
                #print(CAListMst.loc[SelMG, 'WIP Mater Group CM'])
            pass
        #print(CAListMst[CAListMst['CMB_Segment_S1'] == 'CORP']['Assign by TH'])
        #CAListMst.to_excel('IMIS Checking.xlsx')
        #print(CMTeam)
    else:
        WIPRID_Ext = pd.DataFrame(columns=WIPMI.columns.tolist())
        print('Empty List.')

    print('Total WIP IMIS (KYC Ops):\t%d' % ttl_IMIS_WIP)
    KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)
    print(KYCOpsCMTeam)

In [81]:
####### Code for case with presvious assigned IMIS Group #######
####### Checked, Okay #######

if KYC_CM_Exist and np.sum(SelBBRM_KYCOps) > 0:
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel_BRM = CAListMst['BRM Case']
    SelBBRM_KYCOps = np.logical_and(SelBBRM, Sel_BRM)
    #print('No of BBRM cases handled by KYC Ops CM Team:\t%d' % np.sum(SelBBRM_KYCOps))
    
#     SelPrevious = np.logical_not(np.logical_or(NW_Ass_Case['IMISGroupNo'].isnull(), NW_Ass_Case['IMISGroupNo'] == ''))
    SelPrevious = np.logical_not(NW_Ass_Case['IMISGroupNo'].isnull())
    #print(np.sum(SelPrevious))
    if np.sum(SelPrevious) > 0:
#         RelIDwCM = NW_Ass_Case['IMISGroupNo'][SelPrevious].tolist()
        RelIDwCM = NW_Ass_Case['IMISGroupNo'][SelPrevious]
        RelIDwCM = RelIDwCM[RelIDwCM.isin(CAListMst['IMIS Group No'])].tolist()
        for i in range(0, len(RelIDwCM)):
            Sel00 = CAListMst['IMIS Group No'] == RelIDwCM[i]
            Sel01 = CAListMst['Assign by TH'] == ''
            Sel02 = CAListMst['CMB_Segment_S1'] == 'BBRM'
            Sel = np.logical_and(Sel00, Sel01)
            Sel = np.logical_and(Sel, Sel02)
            Sel = np.logical_and(Sel, SelBBRM_KYCOps)
            
            if np.sum(Sel) > 0:
#                 print('No of Case:\t%d' % np.sum(Sel))
                SelIMID = NW_Ass_Case['IMISGroupNo'] == RelIDwCM[i]
                #SelETBCM = NW_Ass_Case[SelIMID]['Assign by TH'] != KYCOps_CMTH
                SelETBCM = np.logical_not(NW_Ass_Case[SelIMID]['Assign by TH'].isin(KYCOps_CMTH))
                tmpCM = NW_Ass_Case[SelIMID]['CM Assigned'].tolist()[0]
                print(RelIDwCM[i])
                print(tmpCM)
                if np.sum(SelETBCM) > 0:
#                     print('With Previous ETB CM')
                    
                    CAListMst.loc[Sel, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
                    if tmpCM == '' or pd.isnull(tmpCM):
                        if np.sum(Sel) > 1:
                            CAListMst.loc[Sel, 'CM Name'] = 'CM' + str(iCM)
                            iCM += 1
                    else:
                        if np.sum(Sel) > 1:
                            CAListMst.loc[Sel, 'CM Name'] = 'CM' + str(iCM)
                            iCM += 1
                        
                    CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (BBRM) (KYC OPS) (tsf to KYC Ops)'
                        
                else:
#                     print('With Previous KYC Ops CM')
                    CAListMst.loc[Sel, 'Assign by TH'] = NW_Ass_Case[SelIMID]['Assign by TH'].tolist()[0]
                    if tmpCM == '' or pd.isnull(tmpCM):
                        CAListMst.loc[Sel, 'CM Name'] = '***CM' + str(iCM)
                        iCM += 1
                    else:
                        CAListMst.loc[Sel, 'CM Name'] = tmpCM
                    CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (BBRM) (KYC Ops)'

            KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)

    else:
        print('Empty List.')
    print(KYCOpsCMTeam)

In [82]:
####### Code for case with non WIP IMIS Group #######
####### Checked, Okay #######
# SelBBRM_KYCOps_C = SelBBRM_KYCOps
if KYC_CM_Exist and np.sum(SelBBRM_KYCOps) > 0:
    
    SelBBRM_KYCOps_C = SelBBRM_KYCOps
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel_BRM = CAListMst['BRM Case']
    SelBBRM_KYCOps = np.logical_and(SelBBRM, Sel_BRM)
    
    IMIS_KYCOps = CAListMst[np.logical_and(SelBBRM_KYCOps, (CAListMst['IMIS Group No'] != ''))]['IMIS Group No'].value_counts().index.tolist()
    
    for aIMIS in IMIS_KYCOps:
        Sel_NoCMTH = CAListMst['Assign by TH'] == ''
        Sel01 = CAListMst['IMIS Group No'] == aIMIS
        Sel = np.logical_and(Sel01, Sel_NoCMTH)
        Sel = np.logical_and(Sel, SelBBRM_KYCOps)
        
        if np.sum(Sel) > 0:
#             print('No of case:\t%d' % np.sum(Sel))
#             print(THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc))
            tmp_name, SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
            CAListMst.loc[Sel, 'Assign by TH'] = ''.join(tmp_name)
#             CAListMst.loc[Sel, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
#             print(SLoc)
            if np.sum(Sel) > 1:
                CAListMst.loc[Sel, 'CM Name'] = 'CM' + str(iCM)
                iCM += 1
                
            CAListMst.loc[Sel, 'Assigment Method'] = 'non WIP (BBRM) (KYC Ops)'
        KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)   
    SelBBRM_KYCOps = SelBBRM_KYCOps_C
#CMTeam


In [83]:
####### Code for case with no IMIS Group #######
####### Checked, Okay #######
# SelBBRM_KYCOps_C = SelBBRM_KYCOps
if KYC_CM_Exist and np.sum(SelBBRM_KYCOps) > 0:
    SelBBRM_KYCOps_C = SelBBRM_KYCOps
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'BBRM'
    Sel_BRM = CAListMst['BRM Case']
    SelBBRM_KYCOps = np.logical_and(SelBBRM, Sel_BRM)
    Sel_NoCMTH = CAListMst['Assign by TH'] == ''
    Sel_NoIMIS = CAListMst['IMIS Group No'] == ''
    
    Sel = np.logical_and(Sel_NoCMTH, SelBBRM_KYCOps)
    Sel = np.logical_and(Sel, Sel_BRM)
    Sel = np.logical_and(Sel, SelBBRM)
    Sel = np.logical_and(Sel, Sel_NoIMIS)
    if np.sum(Sel) > 0:
        for aID in CAListMst[Sel]['Customer_ID_S1'].tolist():
            #print(aID)
            sSel = CAListMst['Customer_ID_S1'] == aID
            CAListMst.loc[sSel, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
            CAListMst.loc[sSel, 'Assigment Method'] = 'Mix BBRM (KYC Ops)'
    
            KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)
    SelBBRM_KYCOps = SelBBRM_KYCOps_C
    print(KYCOpsCMTeam)

# 2.2 Code for Case Assignment (KYC Ops Handled Corp Case)

In [84]:
if KYC_CM_Exist:
    KYCOps_Corp = KYCOps_Corp.fillna('')
    SelCorp = CAListMst['CMB_Segment_S1'] == 'CORP'
    SelCorp_KYCOps = np.repeat(False, CAListMst.shape[0])
    for index, rows in KYCOps_Corp.iterrows():
        
#         SelTeam = CAListMst['RM_Team_Group_S1'] == rows['Team']
        if rows['Sub Team'] ==  '':
            SelSubTeam = CAListMst['RM_Name_S1'] == rows['RM Name']
        else:
            SelSubTeam = CAListMst['RM Sub Team'] == rows['Sub Team']
        Sel = np.logical_and(SelCorp, SelSubTeam)
        if rows['Sub Team'] ==  '':
            print('%s : %s:\t%d' % (rows['Team'], rows['RM Name'], np.sum(Sel)))
        else:
            print('%s : %s:\t%d' % (rows['Team'], rows['Sub Team'], np.sum(Sel)))
        SelCorp_KYCOps = np.logical_or(Sel, SelCorp_KYCOps)
    
    
    SelCorp_KYCOps = np.logical_and(SelCorp, SelCorp_KYCOps)
    print('No of CORP cases handled by KYC Ops CM Team:\t%d' % np.sum(SelCorp_KYCOps))
    CAListMst['KYC Ops Corp Case'] = SelCorp_KYCOps

In [85]:
####### Code for case with WIP Master Group #######


if KYC_CM_Exist and np.sum(SelCorp_KYCOps) > 0:
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'CORP'
    Sel_BRM = CAListMst['KYC Ops Corp Case']
    SelCorp_KYCOps = np.logical_and(SelBBRM, Sel_BRM)

    CorpSel00 = CAListMst['Assign by TH'] == ''
    try:
        CorpSel01 = CAListMst['Master Group ID'] != ''
    except:
        CorpSel01 = CAListMst['Master Group ID'].replace(0, '') != ''
    CorpSel02 = CAListMst['WIP MG']
    
    #print(CAListMst[CAListMst['IMIS Group No'] == 2620]['WIP IMIS ID'])
    CorpSel = np.logical_and(CorpSel00, CorpSel02)
    CorpSel = np.logical_and(CorpSel, CorpSel01)
    CorpSel = np.logical_and(CorpSel, SelCorp_KYCOps)
    print('No of case to be assigned:\t%d' % np.sum(CorpSel))
    ttl_IMIS_WIP = 0
    if np.sum(CorpSel) > 0:
        WIPRID_Ext = WIPMI.copy(deep = True)
        #WIPRID_Ext = WIPRID_Ext[np.logical_not(WIPRID_Ext['IMISGroupNo'].isnull())]
        
        #WIPRID_Ext = WIPRID_Ext[(WIPRID_Ext['IMISGroupNo'] != '')]
        WIPRID_Ext = WIPRID_Ext.sort_values(by='T0', ascending=False)
        WIPRID_Ext = WIPRID_Ext.drop_duplicates(['MasterGroup ID'], keep='first')
#         print(WIPRID_Ext[WIPRID_Ext['IMISGroupNo'] == 2620][['Customer ID', 'Master-CM Name', 'Master-CM Team Head', 'T0']])
        CorpMG = CAListMst[CorpSel]
        #print(CorpMG.columns.tolist())
        CorpMG = pd.merge(CorpMG, WIPRID_Ext[['MasterGroup ID', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='Master Group ID', right_on='MasterGroup ID', suffixes=(False, False))
        #print(CorpMG.shape)
        CorpMG = CorpMG[np.logical_not(CorpMG['T0'].isnull())][['Master Group ID', 'Master-CM Team Head', 'Master-CM Name']]
        #print(CorpMG.shape)
        #CAListMst.loc[CorpSel, ['Assign by TH']] = CorpMG['Master-CM Team Head'].tolist()

        CorpMG = pd.merge(CorpMG, CMTeam[['Skipped']], how='left', left_on='Master-CM Team Head', right_index=True, suffixes=(False, False))
        #CorpMG = CorpMG[np.logical_not(CorpMG['Skipped'])]
        #print(CorpMG.columns)
        #print(CorpMG['IMIS Group No'] == )
#         print(CorpMG.index.tolist())
        for aMGIND in CorpMG.index.tolist():
            #print('IMIS ID:\t%d' % aMGIND)
            SelMG = CAListMst['Master Group ID'] == CorpMG.loc[aMGIND]['Master Group ID']
            #if aMGIND == 47041:
            #    print(np.sum(SelMG))
            ttl_IMIS_WIP += np.sum(SelMG)
            if np.sum(SelMG) > 0:
                #print('Team:\t%s\tCM:\t%s' % (CorpMG.loc[aMGIND]['Master-CM Team Head'], CorpMG.loc[aMGIND]['Master-CM Name']))
#                 print(CorpMG.loc[aMGIND]['Master-CM Team Head'])
                
                if not CorpMG.loc[aMGIND]['Master-CM Team Head'] in (KYCOps_CMTH):
#                 if np.sum(np.logical_not(CorpMG.loc[aMGIND]['Master-CM Team Head'].isin(KYCOps_CMTH))):
#                     print('Re-Assigned.')
                    #print(SLoc)
                    #print(THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc))
                    CAListMst.loc[SelMG, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
                    #print(SLoc)
                    #CAListMst.loc[SelMG, 'Assign by TH'] = KYCOps_CMTH
                    #CAListMst.loc[SelMG, 'CM Name'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'Assigment Method'] = 'WIP (CORP) (KYC Ops) (Tsf WIP to KYC Ops)'
                    if np.sum(SelMG) > 1:
                        CAListMst.loc[SelMG, 'CM Name'] = 'CM' + str(iCM)
                        iCM += 1
                    
                else:
#                     print('Assigned.')
                    CAListMst.loc[SelMG, 'Assign by TH'] = CorpMG.loc[aMGIND]['Master-CM Team Head']
                    #CAListMst.loc[SelMG, 'CM Name'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
                    CAListMst.loc[SelMG, 'Assigment Method'] = 'WIP (CORP) (KYC Ops)'
                    
                #print('-------------------------------------')
                #print(CAListMst.loc[SelMG, 'Assign by TH'])
                #print(CAListMst.loc[SelMG, 'WIP Mater Group CM'])
            pass
        #print(CAListMst[CAListMst['CMB_Segment_S1'] == 'CORP']['Assign by TH'])
        #CAListMst.to_excel('IMIS Checking.xlsx')
        #print(CMTeam)
    else:
        WIPRID_Ext = pd.DataFrame(columns=WIPMI.columns.tolist())
        print('Empty List.')

    print('Total WIP MG (KYC Ops):\t%d' % ttl_IMIS_WIP)
    KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)
    print(KYCOpsCMTeam)

In [86]:
####### Code for case with presvious assigned Master Group #######
####### Checked, Okay #######

# SelCorp_KYCOps_C = SelCorp_KYCOps
if KYC_CM_Exist and np.sum(SelCorp_KYCOps) > 0:
    SelCorp_KYCOps_C = SelCorp_KYCOps
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'CORP'
    Sel_BRM = CAListMst['KYC Ops Corp Case']
    SelCorp_KYCOps = np.logical_and(SelBBRM, Sel_BRM)
    #print('No of BBRM cases handled by KYC Ops CM Team:\t%d' % np.sum(SelBBRM_KYCOps))
    
#     SelPrevious = np.logical_not(np.logical_or(NW_Ass_Case['IMISGroupNo'].isnull(), NW_Ass_Case['IMISGroupNo'] == ''))
    SelPrevious = np.logical_not(NW_Ass_Case['MasterGroup ID'].isnull())
    #print(np.sum(SelPrevious))
    if np.sum(SelPrevious) > 0:
#         RelIDwCM = NW_Ass_Case['MasterGroup ID'][SelPrevious].tolist()
        RelIDwCM = NW_Ass_Case['MasterGroup ID'][SelPrevious]
        RelIDwCM = RelIDwCM[RelIDwCM.isin(CAListMst['Master Group ID'])].tolist()
        for i in range(0, len(RelIDwCM)):
            Sel00 = CAListMst['Master Group ID'] == RelIDwCM[i]
            Sel01 = CAListMst['Assign by TH'] == ''
            Sel02 = CAListMst['CMB_Segment_S1'] == 'CORP'
            Sel = np.logical_and(Sel00, Sel01)
            Sel = np.logical_and(Sel, Sel02)
            Sel = np.logical_and(Sel, SelCorp_KYCOps)
            
            if np.sum(Sel) > 0:
                print('No of Case:\t%d' % np.sum(Sel))
                SelIMID = NW_Ass_Case['MasterGroup ID'] == RelIDwCM[i]
                SelETBCM = np.logical_not(NW_Ass_Case[SelIMID]['Assign by TH'].isin(KYCOps_CMTH))
                tmpCM = NW_Ass_Case[SelIMID]['CM Assigned'].tolist()[0]
                #print(RelIDwCM[i])
                if np.sum(SelETBCM) > 0:
                    print('With Previous ETB CM')
                    tmp_name, SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
                    CAListMst.loc[Sel, 'Assign by TH'] = ''.join(tmp_name)
#                     CAListMst.loc[Sel, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
                    if (tmpCM == '') or pd.isnull(tmpCM):
                        if np.sum(Sel) > 1:
                            CAListMst.loc[Sel, 'CM Name'] = 'CM' + str(iCM)
                            iCM += 1
                    else:
                        if np.sum(Sel) > 1:
                            CAListMst.loc[Sel, 'CM Name'] = 'CM' + str(iCM)
                            iCM += 1
                    CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (CORP) (KYC OPS) (tsf to KYC Ops)'
                        
                else:
                    print('With Previous KYC Ops CM')
                    CAListMst.loc[Sel, 'Assign by TH'] = NW_Ass_Case[SelIMID]['Assign by TH'].tolist()[0]
                    if (tmpCM == '') or pd.isnull(tmpCM):
                        CAListMst.loc[Sel, 'CM Name'] = '***CM' + str(iCM)
                        iCM += 1
                    else:
                        CAListMst.loc[Sel, 'CM Name'] = tmpCM
                    CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (CORP) (KYC Ops)'

            KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)

    else:
        print('Empty List.')
    CMTeam
    
    SelCorp_KYCOps = SelCorp_KYCOps_C
    print(KYCOpsCMTeam)

In [87]:
####### Code for case with non WIP Master Group #######
####### Checked, Okay #######

CaseNo = 0
# SelCorp_KYCOps_C = SelCorp_KYCOps
if KYC_CM_Exist and np.sum(SelCorp_KYCOps) > 0:
    SelCorp_KYCOps_C = SelCorp_KYCOps
    SelBBRM = CAListMst['CMB_Segment_S1'] == 'CORP'
    Sel_BRM = CAListMst['KYC Ops Corp Case']
    SelCorp_KYCOps = np.logical_and(SelBBRM, Sel_BRM)
    
    try:
        IMIS_KYCOps = CAListMst[np.logical_and(SelCorp_KYCOps, (CAListMst['Master Group ID'] != ''))]['Master Group ID'].value_counts().index.tolist()
    except:
        IMIS_KYCOps = CAListMst[np.logical_and(SelCorp_KYCOps, (CAListMst['Master Group ID'].replace(0, '') != ''))]['Master Group ID'].value_counts().index.tolist()
    print(len(IMIS_KYCOps))
    for aIMIS in IMIS_KYCOps:
        #print(aIMIS)
        Sel_NoCMTH = CAListMst['Assign by TH'] == ''
        Sel01 = CAListMst['Master Group ID'] == aIMIS
        Sel = np.logical_and(Sel01, Sel_NoCMTH)
        Sel = np.logical_and(Sel, SelCorp_KYCOps)
        
        if np.sum(Sel) > 0:
            tmp_name, SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
            CAListMst.loc[Sel, 'Assign by TH'] = ''.join(tmp_name)
#             CAListMst.loc[Sel, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
            CaseNo += np.sum(Sel)
            if np.sum(Sel) > 1:
                CAListMst.loc[Sel, 'CM Name'] = 'CM' + str(iCM)
                iCM += 1
                
            CAListMst.loc[Sel, 'Assigment Method'] = 'non WIP (CORP) (KYC Ops)'
        KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst) 
    SelCorp_KYCOps = SelCorp_KYCOps_C
print('No of Case:\t%d' % CaseNo)
print(KYCOpsCMTeam)

No of Case:	0



In [88]:
####### Code for case with no Master Group #######
####### Checked, Okay #######

# SelCorp_KYCOps_C = SelCorp_KYCOps

if KYC_CM_Exist and np.sum(SelCorp_KYCOps) > 0:
    SelCorp_KYCOps_C = SelCorp_KYCOps
    SelCorp = CAListMst['CMB_Segment_S1'] == 'CORP'
    SelSubTeam = CAListMst['KYC Ops Corp Case']
    SelCorp_KYCOps = np.logical_and(SelCorp, SelSubTeam)
    Sel_NoCMTH = CAListMst['Assign by TH'] == ''
    Sel_NoIMIS = CAListMst['Master Group ID'].replace(0, '') == ''
    
    Sel = np.logical_and(Sel_NoCMTH, SelCorp_KYCOps)
    Sel = np.logical_and(Sel, SelSubTeam)
    Sel = np.logical_and(Sel, SelCorp)
    Sel = np.logical_and(Sel, Sel_NoIMIS)
    if np.sum(Sel) > 0:
        print('No of Case:\t%d' % np.sum(Sel))
        print(CAListMst[Sel]['Customer_ID_S1'].tolist())
        for aID in CAListMst[Sel]['Customer_ID_S1'].tolist():
            print(aID)
            sSel = CAListMst['Customer_ID_S1'] == aID
            CAListMst.loc[sSel, 'Assign by TH'], SLoc = THSeqForAssg(KYCOpsCMTeam, 1, 'Capacity', False, SLoc)
            CAListMst.loc[sSel, 'Assigment Method'] = 'Mix Corp (KYC Ops)'
    
            KYCOpsCMTeam = UpdateCapacity(KYCOpsCMTeam, CAListMst)
    SelCorp_KYCOps = SelCorp_KYCOps_C
    print(KYCOpsCMTeam)

In [89]:
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


## Update Capacity Before ETB Team Assignment

In [90]:
CAListMst.loc[CAListMst['Assign by TH'] == 'ETB CM', 'Assigment Method'] = ''
CAListMst['Assign by TH'] = CAListMst['Assign by TH'].replace('ETB CM', '')
CAListMst['CM Name'] = CAListMst['CM Name'].replace('ETB CM', '')

In [91]:
SLoc = 0
SelETBCM = (CMTeam['Segment'] == 'CORP') | (CMTeam['Segment'] == 'BBRM') | (CMTeam['Segment'] == 'CMB')
TTL_Case = 0
ForcedCase_No = np.sum(CAListMst['Assign by TH'].isin(CMTeam['Segment'][SelETBCM]))
Unassigned_No = np.sum(CAListMst['Assign by TH'] == '')
TTL_Case = ForcedCase_No + Unassigned_No
TTL_Case

CMTeam['No of Total Case'] = np.ceil(CMTeam['% HP'] * TTL_Case)
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,0,0,0,1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,0,0,0,1.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,0,0,0,1.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


# 5. Code for Case Assignment with WIP MG

In [92]:
CorpSel00 = CAListMst['Assign by TH'] == ''
CorpSel01 = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')
CorpSel02 = CAListMst['WIP MG']
CorpSel = np.logical_and(CorpSel00, CorpSel01)
CorpSel = np.logical_and(CorpSel, CorpSel02)

if np.sum(CorpSel) > 0:

    WIPMG_Ext = WIPMI.copy(deep = True)
    WIPMG_Ext = WIPMG_Ext[np.logical_not(WIPMG_Ext['MasterGroup Name'].isnull())]
    WIPMG_Ext = WIPMG_Ext.sort_values(by='T0', ascending=False)
    WIPMG_Ext = WIPMG_Ext.drop_duplicates(['MasterGroup Name'], keep='first')

    CorpMG = CAListMst[CorpSel]
    #print(CorpMG.shape)
    CorpMG = pd.merge(CorpMG, WIPMG_Ext[['MasterGroup ID', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='Master Group ID', right_on='MasterGroup ID', suffixes=(False, False))
    #print(CorpMG.shape)
    CorpMG = CorpMG[np.logical_not(CorpMG['T0'].isnull())][['Master Group ID', 'Master-CM Team Head', 'Master-CM Name']]
    #print(CorpMG.shape)
    #CAListMst.loc[CorpSel, ['Assign by TH']] = CorpMG['Master-CM Team Head'].tolist()

    CorpMG = pd.merge(CorpMG, CMTeam[['Skipped']], how='left', left_on='Master-CM Team Head', right_index=True, suffixes=(False, False))
    CorpMG = CorpMG[np.logical_not(CorpMG['Skipped'])]
    #print(CorpMG.columns.tolist())
    for aMGIND in CorpMG.index.tolist():
        SelMG = CAListMst['Master Group ID'] == CorpMG.loc[aMGIND]['Master Group ID']
        #print(CAListMst['Master_Group_ID'].tolist())
        if np.sum(SelMG) > 0:
            CAListMst.loc[SelMG, 'Assign by TH'] = CorpMG.loc[aMGIND]['Master-CM Team Head']
            CAListMst.loc[SelMG, 'CM Name'] = CorpMG.loc[aMGIND]['Master-CM Name']
            CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
            CAListMst.loc[SelMG, 'Assigment Method'] = 'WIP (CORP)'
        pass
    #print(CAListMst[CAListMst['CMB_Segment_S1'] == 'CORP']['Assign by TH'])
    
    #print(CMTeam)
    
else:
    WIPMG_Ext = pd.DataFrame(columns=WIPMI.columns.tolist())
    print('Empty List.')
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

########  Debug Code ########
#CAListMst.to_excel('CM Assg - 05 MG WIP Assigned.xlsx')

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [93]:
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [94]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [95]:
####### Debug Checking #######
# CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 6. Code for Case Assignment with WIP IMIS ID

In [96]:
######## Debug Code ########

#WIPRID_Ext = WIPMI.copy(deep = True)
#WIPRID_Ext = WIPRID_Ext[(WIPRID_Ext['IMISGroupNo'] != '')]
#WIPRID_Ext = WIPRID_Ext.sort_values(by='T0', ascending=False)
#WIPRID_Ext = WIPRID_Ext.drop_duplicates(['IMISGroupNo'], keep='first')
#sel = WIPRID_Ext['IMISGroupNo'] == 25560
#WIPRID_Ext[sel][['Master-CM Name', 'Master-CM Team Head']]

In [97]:
######## Debug Code ########

#WIPMI[WIPMI['IMISGroupNo'] == 20660][['Master-CM Name', 'Master-CM Team Head']]

In [98]:
CorpSel00 = CAListMst['Assign by TH'] == ''
CorpSel01 = CAListMst['CMB_Segment_S1'] == 'BBRM'
CorpSel02 = CAListMst['WIP IMIS ID']
CorpSel = np.logical_and(CorpSel00, CorpSel01)
CorpSel = np.logical_and(CorpSel, CorpSel02)
print(np.sum(CorpSel))
ttl_IMIS_WIP = 0
if np.sum(CorpSel) > 0:

    WIPRID_Ext = WIPMI.copy(deep = True)
    #WIPRID_Ext = WIPRID_Ext[np.logical_not(WIPRID_Ext['IMISGroupNo'].isnull())]
    print(np.sum(WIPRID_Ext['IMISGroupNo'] != ''))
    WIPRID_Ext = WIPRID_Ext[(WIPRID_Ext['IMISGroupNo'] != '')]
    WIPRID_Ext = WIPRID_Ext.sort_values(by='T0', ascending=False)
    WIPRID_Ext = WIPRID_Ext.drop_duplicates(['IMISGroupNo'], keep='first')

    CorpMG = CAListMst[CorpSel]
    #print(CorpMG.columns.tolist())
    CorpMG = pd.merge(CorpMG, WIPRID_Ext[['IMISGroupNo', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='IMIS Group No', right_on='IMISGroupNo', suffixes=(False, False))
    #print(CorpMG.shape)
    CorpMG = CorpMG[np.logical_not(CorpMG['T0'].isnull())][['IMIS Group No', 'Master-CM Team Head', 'Master-CM Name']]
    #print(CorpMG.shape)
    #CAListMst.loc[CorpSel, ['Assign by TH']] = CorpMG['Master-CM Team Head'].tolist()

    CorpMG = pd.merge(CorpMG, CMTeam[['Skipped']], how='left', left_on='Master-CM Team Head', right_index=True, suffixes=(False, False))
    CorpMG = CorpMG[np.logical_not(CorpMG['Skipped'])]
    #print(CorpMG)
    for aMGIND in CorpMG.index.tolist():
        SelMG = CAListMst['IMIS Group No'] == CorpMG.loc[aMGIND]['IMIS Group No']
        if aMGIND == 47041:
            print(np.sum(SelMG))
        ttl_IMIS_WIP += np.sum(SelMG)
        if np.sum(SelMG) > 0:
            #print(CorpMG.loc[aMGIND]['Master-CM Team Head'])
            #print(CorpMG.loc[aMGIND]['Master-CM Name'])
            
            CAListMst.loc[SelMG, 'Assign by TH'] = CorpMG.loc[aMGIND]['Master-CM Team Head']
            CAListMst.loc[SelMG, 'CM Name'] = CorpMG.loc[aMGIND]['Master-CM Name']
            CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
            CAListMst.loc[SelMG, 'Assigment Method'] = 'WIP (BBRM)'
            #print('-------------------------------------')
            #print(CAListMst.loc[SelMG, 'Assign by TH'])
            #print(CAListMst.loc[SelMG, 'WIP Mater Group CM'])
        pass
    #print(CAListMst[CAListMst['CMB_Segment_S1'] == 'CORP']['Assign by TH'])
    #CAListMst.to_excel('IMIS Checking.xlsx')
    #print(CMTeam)
else:
    WIPRID_Ext = pd.DataFrame(columns=WIPMI.columns.tolist())
    print('Empty List.')

print('Total WIP IMIS:\t%d' % ttl_IMIS_WIP)
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

######## Debug Code ########
#CAListMst.to_excel('CM Assg - 06 IMIS WIP Assigned.xlsx')

0
Empty List.
Total WIP IMIS:	0


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [99]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [100]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 3. Code for assigning MG to previous assigned CM Team but T0 has not been issued

In [101]:
SelPrevious = np.logical_not(NW_Ass_Case['MasterGroup ID'].isnull())
if np.sum(SelPrevious) > 0:
    NW_Ass_Case['Latest Assigment Date (MG)'] = NW_Ass_Case['Latest Assigment Date (MG)'].fillna(datetime(2017, 1, 1, 0, 0))
    MGIDwCM = NW_Ass_Case['MasterGroup ID'][SelPrevious].tolist()
    for i in range(0, len(MGIDwCM)):
        Sel00 = CAListMst['Master Group ID'] == MGIDwCM[i]
        Sel01 = CAListMst['Assign by TH'] == ''
        Sel02 = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')
        Sel03 = (NW_Ass_Case['Latest Assigment Date (MG)'] - datetime.now()).dt.days < PreCA_CutDay
        Sel = Sel00 & Sel01 & Sel02 & Sel03
#         Sel = np.logical_and(Sel00, Sel01)
#         Sel = np.logical_and(Sel, Sel02)
        if np.sum(Sel) > 0:
            SelMGID = NW_Ass_Case['MasterGroup ID'] == MGIDwCM[i]
            #if MGIDwCM[i] == 1994670:
            #    print(i)
            #    print(NW_Ass_Case[SelMGID]['Assign by TH'])
            CAListMst.loc[Sel, 'Assign by TH'] = NW_Ass_Case[SelMGID]['Assign by TH'].tolist()[0]
            CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (CORP)'
            
            #CAListMst.loc[Sel, 'CM Name'] = NW_Ass_Case.loc[i]['CM Assigned']
            tmpCM = NW_Ass_Case[SelMGID]['CM Assigned'].tolist()[0]
            if (tmpCM == '') or pd.isnull(tmpCM):
                CAListMst.loc[Sel, 'CM Name'] = '***CM' + str(iCM)
                iCM += 1
            else:
                CAListMst.loc[Sel, 'CM Name'] = tmpCM
            
    CMTeam = UpdateCapacity(CMTeam, CAListMst)
    NW_Ass_Case['Latest Assigment Date (MG)'] = NW_Ass_Case['Latest Assigment Date (MG)'].replace(datetime(2017, 1, 1, 0, 0), pd.NaT)
else:
    print('Empty List.')
CMTeam

########  Debug Code ########
#CAListMst.to_excel('CM Assg - 03 MG Prvious Assigned.xlsx')

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [102]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [103]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 4. Code for assigning IMIS ID to previous assigned CM Team but T0 has not been issued


In [104]:
SelPrevious = np.logical_not(NW_Ass_Case['IMISGroupNo'].isnull())
if np.sum(SelPrevious) > 0:
    NW_Ass_Case['Latest Assigment Date (IMIS)'] = NW_Ass_Case['Latest Assigment Date (IMIS)'].fillna(datetime(2017, 1, 1, 0, 0))
    RelIDwCM = NW_Ass_Case['IMISGroupNo'][SelPrevious].tolist()
    for i in range(0, len(RelIDwCM)):
        Sel00 = CAListMst['IMIS Group No'] == RelIDwCM[i]
        Sel01 = CAListMst['Assign by TH'] == ''
        Sel02 = CAListMst['CMB_Segment_S1'] == 'BBRM'
        Sel03 = (NW_Ass_Case['Latest Assigment Date (IMIS)'] - datetime.now()).dt.days < PreCA_CutDay
        Sel = Sel00 & Sel01 & Sel02 & Sel03
#         Sel = np.logical_and(Sel, Sel02)
        #print(np.sum(Sel))
        if np.sum(Sel) > 0:
            SelIMID = NW_Ass_Case['IMISGroupNo'] == RelIDwCM[i]
            CAListMst.loc[Sel, 'Assign by TH'] = NW_Ass_Case[SelIMID]['Assign by TH'].tolist()[0]
        #print(RelIDwCM[i])
            tmpCM = NW_Ass_Case[SelIMID]['CM Assigned'].tolist()[0]
            
        #if RelIDwCM[i] == 999000002831:
        #    print(np.sum(Sel))
        #    print(NW_Ass_Case.loc[i]['CM Assigned'])

            if tmpCM == '' or pd.isnull(tmpCM):
                CAListMst.loc[Sel, 'CM Name'] = '***CM' + str(iCM)
                CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (BBRM)'
            else:
                #CAListMst.loc[Sel, 'CM Name'] = NW_Ass_Case.loc[i]['CM Assigned']
                CAListMst.loc[Sel, 'CM Name'] = tmpCM
                CAListMst.loc[Sel, 'Assigment Method'] = 'Per Previous Assignment but without T0 (BBRM)'
                #if RelIDwCM[i] == 999000002831:
                #    print("---")
                #    print(CAListMst.loc[Sel, 'CM Name'])
    CMTeam = UpdateCapacity(CMTeam, CAListMst)
    NW_Ass_Case['Latest Assigment Date (IMIS)'] = NW_Ass_Case['Latest Assigment Date (IMIS)'].replace(datetime(2017, 1, 1, 0, 0), pd.NaT)
else:
    print('Empty List.')
CMTeam

########  Debug Code ########
#CAListMst.to_excel('CM Assg - 04 IMIS Prvious Assigned.xlsx')

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [105]:
datetime.now()

datetime.datetime(2026, 8, 17, 12, 49, 19, 665450)

In [106]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [107]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

In [108]:
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


# 7. Code for Case Assignment with non-WIP and unassigned MG ID

In [109]:
CorpSel01 = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')
#CorpSel01 = CAListMst['CMB_Segment_S1'] == 'CORP'
try:
    CorpSel02 = (CAListMst['Master Group ID'] != '')
except:
    try:
        CorpSel02 = (CAListMst['Master Group ID'].replace(0, '') != '')
    except:
        CorpSel02 = (CAListMst['Master Group ID'] > 0)
    
CorpSel03 = CAListMst['Assign by TH'] == ''
CorpSel = np.logical_and(CorpSel01, CorpSel02)
CorpSel = np.logical_and(CorpSel, CorpSel03)
#UAMGID = CAListMst[CorpSel]['Master Group ID'].tolist()


# SLoc = 0 # Control the Start of the Sequence


if np.sum(CorpSel) > 0:
    UAMGID = CAListMst[CorpSel]['Master Group ID'].value_counts().index.tolist()

    #UAMGCM, SLoc = THSeqForAssg(CMTeam, len(UAMGID), 'Capacity', False)

    #for i in range(0, len(UAMGCM)):
    #    CAListMst.loc[CAListMst['Master Group ID'] == UAMGID[i], 'Assign by TH'] = UAMGCM[i]
    #    CMTeam = UpdateCapacity(CMTeam, CAListMst)
    #    pass
    #CMTeam = UpdateCapacity(CMTeam, CAListMst)

    #i = 0
    #SLoc = 0 # Control the Start of the Sequence
    for i in range(0, len(UAMGID)):
        #ncMG = np.sum(CAListMst['Master Group ID'] == UAMGID[i])
        #print(THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc))
        SelCase = CAListMst['Master Group ID'] == UAMGID[i]
        SelCase = np.logical_and(SelCase, CorpSel03)
        tmpSLoc = SLoc
        tmpCMTH, SLoc = THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)
        tmpCMTH = tmpCMTH[0]
#         print('Tmp CMTH:\t%s' % tmpCMTH)

        if CheckCapacity(np.sum(SelCase), tmpCMTH, CMTeam) == 0:
            CAListMst.loc[SelCase, 'Assign by TH'] = tmpCMTH
            if np.sum(SelCase) > 1:
                CAListMst.loc[SelCase, 'CM Name'] = 'CM' + str(iCM)
                iCM += 1
        else:
            tmp_CMTH = LocateMaxCapacity(np.sum(SelCase), CMTeam)
            
            if tmp_CMTH == 0:
                CAListMst.loc[SelCase, 'Assign by TH'] = tmpCMTH
                if np.sum(SelCase) > 1:
                    CAListMst.loc[SelCase, 'CM Name'] = 'CM' + str(iCM)
                    iCM += 1
            else:
                CAListMst.loc[SelCase, 'Assign by TH'] = tmp_CMTH
                tmpCMTH = tmp_CMTH
                if np.sum(SelCase) > 1:
                    CAListMst.loc[SelCase, 'CM Name'] = 'CM' + str(iCM)
                    iCM += 1
                SLoc = tmpSLoc
#         print('Actual CMTH:\t%s' % tmpCMTH)
        # CAListMst.loc[SelCase, 'Assign by TH'], SLoc = THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)
        CAListMst.loc[SelCase, 'Assigment Method'] = 'non WIP (CORP)'
        CMTeam = UpdateCapacity(CMTeam, CAListMst)

        pass
else:
    print('Empty List')
CMTeam
######## Debug Code ########
#CAListMst.to_excel('CM Assg - 07 MG Unassigned.xlsx')

Empty List


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [110]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [111]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 8. Code for Case Assignment with non-WIP and unassigned IMIS ID

In [112]:
CorpSel01 = CAListMst['CMB_Segment_S1'] == 'BBRM'
#CorpSel01 = CAListMst['CMB_Segment_S1'] == 'CORP'
CorpSel02 = (CAListMst['IMIS Group No'] != '')
CorpSel03 = CAListMst['Assign by TH'] == ''
CorpSel = np.logical_and(CorpSel01, CorpSel02)
CorpSel = np.logical_and(CorpSel, CorpSel03)

In [113]:
SelID = CAListMst['Customer_ID_S1'] == 'C188161'
np.sum(np.logical_and(SelID, CAListMst['IMIS Group No'] != ''))

0

In [114]:
CorpSel01 = CAListMst['CMB_Segment_S1'] == 'BBRM'
#CorpSel01 = CAListMst['CMB_Segment_S1'] == 'CORP'
CorpSel02 = (CAListMst['IMIS Group No'] != '')
CorpSel03 = CAListMst['Assign by TH'] == ''
CorpSel = np.logical_and(CorpSel01, CorpSel02)
CorpSel = np.logical_and(CorpSel, CorpSel03)
#UAMGID = CAListMst[CorpSel]['Master Group ID'].tolist()

if np.sum(CorpSel) > 0:
    UAMGID = CAListMst[CorpSel]['IMIS Group No'].value_counts().index.tolist()

    #UAMGCM, SLoc = THSeqForAssg(CMTeam, len(UAMGID), 'Capacity', False)

    #for i in range(0, len(UAMGCM)):
    #    CAListMst.loc[CAListMst['Master Group ID'] == UAMGID[i], 'Assign by TH'] = UAMGCM[i]
    #    CMTeam = UpdateCapacity(CMTeam, CAListMst)
    #    pass
    #CMTeam = UpdateCapacity(CMTeam, CAListMst)

    #i = 0
    #SLoc = 0 # Control the Start of the Sequence
    for i in range(0, len(UAMGID)):
        #ncMG = np.sum(CAListMst['Master Group ID'] == UAMGID[i])
        #print(THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc))

        SelCase = CAListMst['IMIS Group No'] == UAMGID[i]        
        SelCase = np.logical_and(SelCase, CorpSel03)
        tmpSLoc = SLoc
        tmpCMTH, SLoc = THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)
        tmpCMTH = tmpCMTH[0]
#         print('Tmp CMTH:\t%s' % tmpCMTH)
        if CheckCapacity(np.sum(SelCase), tmpCMTH, CMTeam) == 0:
            CAListMst.loc[SelCase, 'Assign by TH'] = tmpCMTH
            if np.sum(SelCase) > 1:
                CAListMst.loc[SelCase, 'CM Name'] = 'CM' + str(iCM)
                iCM += 1
        else:
            tmp_CMTH = LocateMaxCapacity(np.sum(SelCase), CMTeam)
            if tmp_CMTH == 0:
                CAListMst.loc[SelCase, 'Assign by TH'] = tmpCMTH
                if np.sum(SelCase) > 1:
                    CAListMst.loc[SelCase, 'CM Name'] = 'CM' + str(iCM)
                    iCM += 1
            else:
                CAListMst.loc[SelCase, 'Assign by TH'] = tmp_CMTH
                tmpCMTH = tmp_CMTH
                if np.sum(SelCase) > 1:
                    CAListMst.loc[SelCase, 'CM Name'] = 'CM' + str(iCM)
                    iCM += 1
                SLoc = tmpSLoc
#         print('Actual CMTH:\t%s' % tmpCMTH)
        # CAListMst.loc[SelCase, 'Assign by TH'], SLoc = THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)
        CAListMst.loc[SelCase, 'Assigment Method'] = 'non WIP (BBRM)'
        CMTeam = UpdateCapacity(CMTeam, CAListMst)

        pass
else:
    print('Empty List')

CMTeam


######## Debug Code ########
#CAListMst.to_excel('CM Assg - 08 IMIS Unassigned.xlsx')

Empty List


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [115]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [116]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 9. Code for non-MG or non-IMIS ID Case Assignment

In [117]:
CorpSel01 = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'BBRM') | (CAListMst['CMB_Segment_S1'] == 'CMB')
#CorpSel01 = CAListMst['CMB_Segment_S1'] == 'CORP'
CorpSel02 = CAListMst['Assign by TH'] == ''
CorpSel = np.logical_and(CorpSel01, CorpSel02)
CorpNoNMG = CAListMst[CorpSel]
CaseList = CorpNoNMG['Customer_ID_S1'].tolist()

if len(CaseList) > 0:
    #SLoc = 0 # Control the Start of the Sequence
    for aID in CaseList:
        #print(aID)
        #print(THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc))
        sSel = CAListMst['Customer_ID_S1'] == aID
        #print('\t01')
        #print(THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)[0])
        CAListMst.loc[sSel, 'Assign by TH'], SLoc = THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)
        CAListMst.loc[sSel, 'Assigment Method'] = 'Mix'
        #print(CAListMst)
        CMTeam = UpdateCapacity(CMTeam, CAListMst)
        pass
print('OK')

CMTeam


OK


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


In [118]:
pd.pivot_table(CAListMst, index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [119]:
print("No of unassigned case:\t%d" % len(CAListMst[CAListMst['Assign by TH'] == '']['CMB_Segment_S1'].tolist()))

#len(CAListMst[CAListMst['CMB_Segment_S1'] == 'BBRM']['CMB_Segment_S1'].tolist())

No of unassigned case:	0


In [120]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]

# 10. Code for BB Case Assignment

In [121]:
CorpSel01 = CAListMst['CMB_Segment_S1'] == 'BBRM'
CorpSel02 = CAListMst['Assign by TH'] == ''
CorpSel = np.logical_and(CorpSel01, CorpSel02)

print(np.sum(CorpSel))

if np.sum(CorpSel) != 0:
    
    #CAListMst.loc[CorpSel, ['Assign by TH']], SLoc = THSeqForAssg(CMTeam, CAListMst[CorpSel].shape[0], 'Capacity', False, SLoc)
    SLoc = 0 # Control the Start of the Sequence
    for aID in CAListMst[CorpSel]['Customer_ID_S1'].tolist():
        sSel = CAListMst['Customer_ID_S1'] == aID
        CAListMst.loc[sSel, 'Assign by TH'], SLoc = THSeqForAssg(CMTeam, 1, 'Capacity', False, SLoc)
        CAListMst.loc[sSel, 'Assigment Method'], SLoc = 'End'

        CMTeam = UpdateCapacity(CMTeam, CAListMst)
        pass
else:
    print('No unassigned case.')
    

#print(SLoc)
#print(CAListMst.shape)

#print(CAListMst['CMB_Segment_S1'].value_counts())
CMTeam = UpdateCapacity(CMTeam, CAListMst)
CMTeam

0
No unassigned case.


,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0


## Process CMTeam DF

In [122]:
CMTeam['Case per FTE'] = CMTeam['Total No of Assigned'] / CMTeam['FTE']

In [123]:
####### Debug Checking #######
#CAListMst[CAListMst['IMIS Group No'] == 587][['Assign by TH', 'CM Name']]
CMTeam_C = CMTeam.copy(deep=True)

In [124]:
if KYC_CM_Exist:
    for aKYCTH in KYCOps_CMTH:
        CMTeam.loc[aKYCTH] = KYCOpsCMTeam.loc[aKYCTH]
        CMTeam.loc[aKYCTH, 'Skipped'] = True
CMTeam['Case per FTE'] = CMTeam['Total No of Assigned'] / CMTeam['FTE']
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity,Case per FTE
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0,0.133333
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0,0.100000
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0,0.000000
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0,0.100000
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0,0.000000
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0,0.000000


In [125]:
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity,Case per FTE
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0,0.133333
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0,0.100000
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0,0.000000
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0,0.100000
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0,0.000000
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0,0.000000


# Update WIP MG CM Column

In [126]:
CorpSel01 = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')
#CorpSel01 = np.logical_or(CAListMst['CMB_Segment_S1'] == 'CORP', CAListMst['CMB_Segment_S1'] == 'BBRM')
CorpSel02 = CAListMst['WIP MG']
CorpSel = np.logical_and(CorpSel02, CorpSel01)

CorpMG = CAListMst[CorpSel]
try:
    CorpMG = pd.merge(CorpMG, WIPMG_Ext[['MasterGroup ID', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='Master Group ID', right_on='MasterGroup ID', suffixes=(False, False))
except ValueError:
    WIPMG_Ext['MasterGroup ID'] = WIPMG_Ext['MasterGroup ID'].astype('int64')
    CorpMG = pd.merge(CorpMG, WIPMG_Ext[['MasterGroup ID', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='Master Group ID', right_on='MasterGroup ID', suffixes=(False, False))

CorpMG = CorpMG[np.logical_not(CorpMG['T0'].isnull())][['Master Group ID', 'Master-CM Team Head', 'Master-CM Name']]
ttlWIP = 0
for aMGIND in CorpMG.index.tolist():
    SelMG = CAListMst['Master Group ID'] == CorpMG.loc[aMGIND]['Master Group ID']
    #print(CAListMst['Master_Group_ID'].tolist())
    ttlWIP += np.sum(SelMG)
    if np.sum(SelMG) > 0:
        CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
    pass
print('No of case:\t%d' % ttlWIP)

No of case:	4


# Update WIP IMIS ID CM Column

In [127]:
CorpSel01 = CAListMst['CMB_Segment_S1'] == 'BBRM'
#CorpSel01 = np.logical_or(CAListMst['CMB_Segment_S1'] == 'CORP', CAListMst['CMB_Segment_S1'] == 'BBRM')
CorpSel02 = CAListMst['WIP IMIS ID']
CorpSel = np.logical_and(CorpSel02, CorpSel01)

CorpMG = CAListMst[CorpSel]
CorpMG = pd.merge(CorpMG, WIPRID_Ext[['IMISGroupNo', 'Master-CM Team Head', 'Master-CM Name', 'T0']], how='left', left_on='IMIS Group No', right_on='IMISGroupNo', suffixes=(False, False))
CorpMG = CorpMG[np.logical_not(CorpMG['T0'].isnull())][['IMISGroupNo', 'Master-CM Team Head', 'Master-CM Name']]
ttlWIP = 0
for aMGIND in CorpMG.index.tolist():
    SelMG = CAListMst['IMIS Group No'] == CorpMG.loc[aMGIND]['IMISGroupNo']
    #print(CAListMst['Master_Group_ID'].tolist())
    ttlWIP += np.sum(SelMG)
    if np.sum(SelMG) > 0:
        CAListMst.loc[SelMG, 'WIP Mater Group CM'] = CorpMG.loc[aMGIND]['Master-CM Name']
    pass
print('No of case:\t%d' % ttlWIP)

No of case:	0


# Updated Last Completion Info

In [128]:
#'Customer_ID_S1', 'Last CM (ref only)', 'Last CMTH (ref only)'
#'Customer ID', 'Master-CM Name', 'Master-CM Team Head'

ComCID = ComMI[ComMI['CIN'].isin(CAListMst['Customer_CIN_S1'].tolist())]['CIN']

for aCID in ComCID:
    #print(aCID)
    SelCID = CAListMst['Customer_CIN_S1'] == aCID
    SelCID01 = ComMI['CIN'] == aCID
    if np.sum(SelCID) > 0:
        CAListMst.loc[SelCID, 'Last CMTH (ref only)'] = ComMI[SelCID01]['Master-CM Team Head']
        CAListMst.loc[SelCID, 'Last CM (ref only)'] = ComMI[SelCID01]['Master-CM Name']
    

# Statistic After Case Assignment

In [129]:
#pickCM = CAListMst['Assign by TH'].isin(MS90CMTeam['MG CM'].value_counts().index.tolist())
pickCM = CAListMst['Assign by TH'] == MGCSM_TH
pd.pivot_table(CAListMst[np.logical_not(pickCM)], index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)

,Customer_CIN_S1
CMB_Segment_S1,CORP
Assign by TH,
JOHN G X LONG,1
KEN M J PAN,1
SUE S B HUANG,2


In [130]:
CMPort = pd.pivot_table(CAListMst[np.logical_not(pickCM)], index=['CMB_Segment_S1', 'RM_Team_Group_S1'], columns='Assign by TH', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)
CMPID = CMPort.index.tolist()
CMPCol = CMPort.columns.tolist()

TTL = []

for aCol in CMPCol:
    CaseT = np.sum(CMPort.loc[:][aCol])
    TTL.append(CaseT)
        


CMPort.loc[('Total', 'Total'), :] = TTL

CMPort

Customer_CIN_S1                          
Assign by TH                      JOHN G X LONG KEN M J PAN SUE S B HUANG
CMB_Segment_S1 RM_Team_Group_S1                                          
CORP           CRF                          1.0         1.0           2.0
Total          Total                        1.0         1.0           2.0

In [131]:
MGTeamStat = pd.pivot_table(CAListMst[pickCM], index='Assign by TH', columns='CMB_Segment_S1', values=['Customer_CIN_S1'] , aggfunc='count', fill_value=0)
BBT = -1
CPT = -1
if MGTeamStat.shape[0] > 0:
    if ('Customer_CIN_S1', 'BBRM') in MGTeamStat.columns.tolist():
        BBT = np.sum(MGTeamStat.loc[:, ('Customer_CIN_S1', 'CORP')])
    if ('Customer_CIN_S1', 'CORP') in MGTeamStat.columns.tolist():
        CPT = np.sum(MGTeamStat.loc[:, ('Customer_CIN_S1', 'CORP')])
    if BBT == -1 and CPT == -1:
        pass
    elif BBT == -1:
        MGTeamStat.loc['Total', :] = [CPT]
    elif CPT == -1:
        MGTeamStat.loc['Total', :] = [BBT]
    else:
        MGTeamStat.loc['Total', :] = [BBT, CPT]
else:
    print('No case for MG Team.')
MGTeamStat

No case for MG Team.


Assign by TH


In [132]:
CMTeam = CMTeam.sort_index()
CMTeam = CMTeam.sort_values('Segment', ascending=False)
CMTeam

,Segment,FTE,Remarks,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Skipped,% HP,No of Total Case,Assign Seq,Total No of BBRM Assigned,Total No of Corp Assigned,Total No of CMB Assigned,Total No of Assigned,Capacity,Case per FTE
KEN Y K FONG,PCSM,11,NaN,NaN,NaN,NaN,0,True,0.000000,0.0,4,0,0,0,0,0.0,0.000000
JOHN G X LONG,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,1,0,1,0,1,0.0,0.100000
KEN M J PAN,CORP,10,2 support out,10.0,915.0,0.166667,152,False,0.166667,1.0,3,0,1,0,1,0.0,0.100000
LILY1 Y L DU,CORP,10,NaN,10.0,915.0,0.166667,152,False,0.166667,1.0,5,0,0,0,0,1.0,0.000000
SUE S B HUANG,CORP,15,"1 ML,2 support out",15.0,915.0,0.250000,230,False,0.250000,1.0,0,0,2,0,2,-1.0,0.133333
YAN J X YAN,CORP,15,1 support out,15.0,915.0,0.250000,229,False,0.250000,1.0,2,0,0,0,0,1.0,0.000000


# Code for Export

### Export Case Assignment Result

In [133]:
# with pd.ExcelWriter('./CMTeam_Loading_' + datetime.now().strftime("%Y%m%d") + '.xlsx') as CA_Stat:
#     CMTeam.to_excel(CA_Stat, sheet_name='CM Team Loading')
#     CMPort.to_excel(CA_Stat, sheet_name='Seg Team Distribute')
#     pd.pivot_table(CAListMst_Ext, values='Customer_ID_S1', index=['Assign by TH'],
#                     columns=['Assigment Method'], aggfunc=len, fill_value=0).to_excel(CA_Stat, sheet_name='CA Method')

In [134]:
ColCAListMst += ['Assigment Method']
try:
    CAListMst['Case Handled by KYC Ops'] = CAListMst['Assign by TH'].isin(KYCOpsCMTeam.index)
except:
    CAListMst['Case Handled by KYC Ops'] = False
CAListMst = CAListMst.sort_values('Org_Seq')
try:
    ColCAListMstKYC = ColCAListMst + ['KYC Ops Corp Case']
    if not ('Case Handled by KYC Ops' in ColCAListMstKYC):
        ColCAListMstKYC = ColCAListMst + ['Case Handled by KYC Ops']
    CAListMst_Ext = CAListMst[ColCAListMstKYC].copy(deep=True)
except KeyError:
#     pass
# CAListMst = CAListMst.sort_values('Org_Seq')
    CAListMst_Ext = CAListMst[ColCAListMst].copy(deep=True)


CAListMst_Ext.to_excel('./CM_Assign.xlsx', index=False)

with pd.ExcelWriter('./CMTeam_Loading_' + datetime.now().strftime("%Y%m%d") + '.xlsx') as CA_Stat:
    CMTeam.to_excel(CA_Stat, sheet_name='CM Team Loading')
    CMPort.to_excel(CA_Stat, sheet_name='Seg Team Distribute')
    pd.pivot_table(CAListMst_Ext, values='Customer_ID_S1', index=['Assign by TH'],
                    columns=['Assigment Method'], aggfunc=len, fill_value=0).to_excel(CA_Stat, sheet_name='CA Method')
#CMTeam.to_excel('./CMTeam_Loading_' + datetime.now().strftime("%Y%m%d") + '.xlsx', index=True)
#CMPort.to_excel('./RMTeam.xlsx', index=True)
#CMTeam_Old = pd.read_excel('./CMTeam.xlsx', index_col=0)
#OldSeq = CMTeam_Old['Assign Seq']
#OldSeq
#GetOldImportSeq(CMTeam)

### WIP MG Statistic

In [135]:
if 'Latest Assigment Date (MG)' in NW_Ass_Case.columns.tolist():
    pass
else:
    NW_Ass_Case['Latest Assigment Date (MG)'] = pd.NaT
    
if 'Latest Assigment Date (IMIS)' in NW_Ass_Case.columns.tolist():
    pass
else:
    NW_Ass_Case['Latest Assigment Date (IMIS)'] = pd.NaT

In [136]:
######## Get CM Team / CM Data from current Case Assignment File #########
accMGAs = 0
if np.sum((CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')) > 0:
    CAListMst_CORP = CAListMst[(CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')]
#     print(np.sum(CAListMst_CORP['Master Group ID'].replace(0, '') != ''))
    try:
        accMGAs = pd.pivot_table(CAListMst_CORP[CAListMst_CORP['Master Group ID'] != ''], index=['Master Group ID'], values='Customer_ID_S1', aggfunc='count', fill_value=0)
    except:
        accMGAs = pd.pivot_table(CAListMst_CORP[CAListMst_CORP['Master Group ID'].replace('', 0) > 0], index=['Master Group ID'], values='Customer_ID_S1', aggfunc='count', fill_value=0)
    #accMGAs.columns = ['No fo Case to be assigned']
    #accMGAs = accMGAs.reset_index()
    #print(accMGAs.columns.tolist())
    
    if accMGAs.shape[0] > 0:
        accMGAs.columns = ['No fo Case to be assigned']
        accMGAs['Latest Assigment Date (MG)'] = datetime.now() 
        accMGAs = accMGAs.reset_index()
    else:
        accMGAs = 0


######## Get CM Team / CM Data from Master MI #########


WIPMG = WIPMI_T.copy(deep=True)
Sel01 = (WIPMG['Segment'] == 'CORP') | (WIPMG['Segment'] == 'CMB')
Sel02 = np.logical_not(WIPMG['MasterGroup ID'].isnull())
Sel03 = WIPMG['MasterGroup ID'] != ''
Sel = np.logical_and(Sel01, Sel02)
Sel = np.logical_and(Sel, Sel03)
WIPMG = WIPMG[Sel]
WIPMG
#aExt = WIPMG[WIPMG['MasterGroup ID'] != ''][['MasterGroup ID', 'MasterGroup Name']].drop_duplicates(subset='MasterGroup ID')
aExt = pd.pivot_table(WIPMG, index=['MasterGroup ID'], values = 'CIN', aggfunc='count', fill_value=0)
aExt.columns = ['No of WIP Case']
aExt = aExt.reset_index()
#aExt
#print(aExt.columns.tolist())


######## Merge CM Team / CM Data from Master MI with MGCSM Full List #########


MGFL_C = MGFL.copy(deep=True)
MGFL_C = MGFL_C.drop_duplicates(subset='Master_Group_ID')
aExt = pd.merge(aExt, MGFL_C[['Master_Group_ID']], how='left', left_on='MasterGroup ID', right_on='Master_Group_ID', suffixes=(False, False))

aExt = aExt.drop(['Master_Group_ID'], axis=1)
aExt = aExt[['MasterGroup ID', 'No of WIP Case']]
#aExt


######## Further Merge CM Team / CM Data with current Case Assignment File #########

#print(aExt.shape[0])
if not 'int' in str(type(accMGAs)) :
    MG_Stat = pd.merge(aExt, accMGAs, how='outer', left_on='MasterGroup ID', right_on='Master Group ID', suffixes=(False, False))
    Sel01 = MG_Stat['MasterGroup ID'].isnull()
    Sel02 = MG_Stat['Master Group ID'].isnull()
    MG_Stat.loc[Sel01, 'MasterGroup ID'] = MG_Stat[Sel01]['Master Group ID']
    MG_Stat = MG_Stat.drop('Master Group ID', axis=1)
        
    
else:
    MG_Stat = aExt
    MG_Stat['Latest Assigment Date (MG)'] = pd.NaT
    MG_Stat['No fo Case to be assigned'] = 0


######## Further Merge CM Team / CM Data with current Case Assignment File #########


MG_Stat = MG_Stat.fillna(0)
MG_Stat['Total'] = MG_Stat['No of WIP Case'] + MG_Stat['No fo Case to be assigned']
MG_Stat['MG CM Team'] = MG_Stat['MasterGroup ID'].isin(MS90CMTeam['Master Group ID'].tolist())
MG_Stat = pd.merge(MG_Stat, MGFL_C[['Master_Group_ID', 'Master_Group_Name']], how='left', left_on='MasterGroup ID', right_on='Master_Group_ID', suffixes=(False, False))
MG_Stat = MG_Stat.drop('Master_Group_ID', axis=1)
#print(MG_Stat.columns.tolist())
MG_Stat = MG_Stat[['MasterGroup ID', 'Master_Group_Name', 'No of WIP Case', 'No fo Case to be assigned', 'Total', 'MG CM Team', 'Latest Assigment Date (MG)']]
try:
    CAListMst_MG_tmp = CAListMst[np.logical_and(CAListMst['Master Group ID'] != '', ((CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')))]
except:
    CAListMst_MG_tmp = CAListMst[np.logical_and(CAListMst['Master Group ID'].replace('', 0) > 0, ((CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')))]
    
CAListMst_MG_tmp = CAListMst_MG_tmp.drop_duplicates(subset=['Master Group ID', 'Assign by TH'])
MG_Stat = pd.merge(MG_Stat, CAListMst_MG_tmp[['Master Group ID', 'Assign by TH']], how='left', left_on='MasterGroup ID', right_on='Master Group ID', suffixes=(False, False))
MG_Stat = MG_Stat[['MasterGroup ID', 'Master_Group_Name', 'No of WIP Case', 'No fo Case to be assigned', 'Total', 'MG CM Team', 'Assign by TH', 'Latest Assigment Date (MG)']]
MG_Stat['CM Assigned'] = ''

PASel01 = MG_Stat['No of WIP Case'] == 0 # New Appeared MG Group
PASel02 = MG_Stat['No fo Case to be assigned'] > 0
PASel = np.logical_and(PASel01, PASel02)
PASel = np.logical_and(PASel, np.logical_not(MG_Stat['MG CM Team']))

MG_Stat['Latest Assigment Date (IMIS)'] = pd.NaT

# if 'Latest Assigment Date (MG)' in NW_Ass_Case.columns.tolist():
#     pass
# else:
#     NW_Ass_Case['Latest Assigment Date (MG)'] = pd.NaT

tmpCol = NW_Ass_Case.columns.tolist()

#tmpCol.remove('IMIS Group No')
tmpCol.remove('IMISGroupNo')
#print(tmpCol)
if 'IMIS Group No.1' in tmpCol:
    tmpCol.remove('IMIS Group No.1')
tmpCol.remove('IMIS Group Name')
#print(tmpCol)
#print(MG_Stat.columns.tolist())


NW_Ass_Case_New_MG = MG_Stat[PASel][tmpCol]


######## NW_Ass_Case is the file storing case without T0 but complete case assignment ########

######## Merging record with case without T0 but complete case assignment previously ######## 


SelNWMG = np.logical_not(NW_Ass_Case['MasterGroup ID'].isin(NW_Ass_Case_New_MG['MasterGroup ID']))
if np.sum(SelNWMG) > 0:
    NW_Ass_Case_New_MG = NW_Ass_Case_New_MG.append(NW_Ass_Case[SelNWMG], ignore_index=True)
NW_Ass_Case_New_MG = NW_Ass_Case_New_MG[np.logical_not(NW_Ass_Case_New_MG['MasterGroup ID'].isnull())]
NW_Ass_Case_New_MG =NW_Ass_Case_New_MG[tmpCol]   

### Relationship ID Statistic

In [137]:
np.sum(CAListMst['CMB_Segment_S1']  == 'BBRM')

0

In [138]:
accRIDs = 0
#print(np.sum(CAListMst['CMB_Segment_S1'] == 'BBRM'))
if np.sum(CAListMst['CMB_Segment_S1'] == 'BBRM') > 0:

    CAListMst_BB = CAListMst[CAListMst['CMB_Segment_S1'] == 'BBRM']
    accRIDs = pd.pivot_table(CAListMst_BB[CAListMst_BB['IMIS Group No'] != ''], index=['IMIS Group No', 'IMIS Group'], values='Customer_ID_S1', aggfunc='count', fill_value=0)
    if accRIDs.shape[0] > 0:
        accRIDs.columns = ['No fo Case to be assigned']
        accRIDs = accRIDs.reset_index()
        accRIDs['Latest Assigment Date (IMIS)'] = datetime.now() 
    else:
        accRIDs = 0

#print(accRIDs)
WIPRID = WIPMI_T.copy(deep=True)
WIPRID = WIPRID[WIPRID['Segment'] == 'BBRM']
#aExt = WIPMG[WIPMG['MasterGroup ID'] != ''][['MasterGroup ID', 'MasterGroup Name']].drop_duplicates(subset='MasterGroup ID')
aExtRID = pd.pivot_table(WIPRID, index=['IMISGroupNo', 'IMIS Group Name', 'Master-CM Team Head'], values = 'CIN', aggfunc='count', fill_value=0)
aExtRID.columns = ['No of WIP Case']
aExtRID = aExtRID.reset_index()



#aExtRID.columns = ['RelationshipID', 'RelationshipID Group Name', 'Assign by TH', 'No of WIP Case']

##print(aExt.shape[0])
#MGFL_C = MGFL.copy(deep=True)
#MGFL_C = MGFL_C.drop_duplicates(subset='Master_Group_ID')
#aExt = pd.merge(aExt, MGFL_C[['Master_Group_ID']], how='left', left_on='MasterGroup ID', right_on='Master_Group_ID', suffixes=(False, False))

###print(aExt.shape[0])
#aExt = aExt.drop(['Master_Group_ID'], axis=1)
#aExt = aExt[['MasterGroup ID', 'No of WIP Case']]

if not 'int' in str(type(accRIDs)) :
#if accRIDs != 0:
#     try:
#         RID_Stat = pd.merge(aExtRID, accRIDs, how='outer', left_on='IMISGroupNo', right_on='IMIS Group No', suffixes=(False, False))
#     except ValueError:
#         aExtRID['IMISGroupNo'] = pd.to_numeric(aExtRID['IMISGroupNo'])
#         RID_Stat = pd.merge(aExtRID, accRIDs, how='outer', left_on='IMISGroupNo', right_on='IMIS Group No', suffixes=(False, False))
    
    aExtRID['IMISGroupNo'] = pd.to_numeric(aExtRID['IMISGroupNo'])
    RID_Stat = pd.merge(aExtRID, accRIDs, how='outer', left_on='IMISGroupNo', right_on='IMIS Group No', suffixes=(False, False))
    
    Sel01 = RID_Stat['IMISGroupNo'].isnull()
    Sel02 = RID_Stat['IMIS Group No'].isnull()
    RID_Stat.loc[Sel01, 'IMISGroupNo'] = RID_Stat[Sel01]['IMIS Group No']
    RID_Stat = RID_Stat.drop('IMIS Group No', axis=1)
    Sel01 = RID_Stat['IMIS Group Name'].isnull()
    RID_Stat.loc[Sel01, 'IMIS Group Name'] = RID_Stat[Sel01]['IMIS Group']
    RID_Stat = RID_Stat.drop('IMIS Group', axis=1)

    
else:
    RID_Stat = aExtRID
    RID_Stat['Latest Assigment Date (IMIS)'] = pd.NaT
    RID_Stat['No fo Case to be assigned'] = 0




RID_Stat = RID_Stat.fillna(0)
RID_Stat['Total'] = RID_Stat['No of WIP Case'] + RID_Stat['No fo Case to be assigned']


#print(RID_Stat.head())

#RID_Stat
#MG_Stat['MG CM Team'] = MG_Stat['MasterGroup ID'].isin(MS90CMTeam['Master Group ID'].tolist())
#MG_Stat = pd.merge(MG_Stat, MGFL_C[['Master_Group_ID', 'Master_Group_Name']], how='left', left_on='MasterGroup ID', right_on='Master_Group_ID', suffixes=(False, False))
#MG_Stat = MG_Stat.drop('Master_Group_ID', axis=1)

#RID_Stat = RID_Stat[['RelationshipID', 'RelationshipID Group Name', 'No of WIP Case', 'No fo Case to be assigned', 'Total']]

CAListMst_RID_tmp = CAListMst[np.logical_and(CAListMst['IMIS Group No'] != '', CAListMst['CMB_Segment_S1'] == 'BBRM')]
CAListMst_RID_tmp = CAListMst_RID_tmp.drop_duplicates(subset=['IMIS Group No', 'Assign by TH'])
RID_Stat = pd.merge(RID_Stat, CAListMst_RID_tmp[['IMIS Group No', 'Assign by TH']], how='left', left_on='IMISGroupNo', right_on='IMIS Group No', suffixes=(False, False))
SelRID = RID_Stat['Assign by TH'].isnull()
RID_Stat.loc[SelRID, 'Assign by TH'] = RID_Stat.loc[SelRID, 'Master-CM Team Head']

RID_Stat = RID_Stat[['IMISGroupNo', 'IMIS Group Name', 'No of WIP Case', 'No fo Case to be assigned', 'Total', 'Assign by TH', 'Latest Assigment Date (IMIS)']]

RID_Stat['Latest Assigment Date (MG)'] = pd.NaT

#MGSel01 = MG_Stat['No fo Case to be assigned'] > 3
#MGSel02 = MG_Stat['No of WIP Case'] > 0
#MGSel03 = np.logical_not(MG_Stat['MG CM Team'])

#MGSel = np.logical_and(MGSel01, MGSel02)
#MGSel = np.logical_and(MGSel, MGSel03)

#MG_Stat[MGSel].sort_values(by=['Total', 'No fo Case to be assigned'], ascending=False)

PASel01 = RID_Stat['No of WIP Case'] == 0
PASel02 = RID_Stat['No fo Case to be assigned'] > 0
PASel = np.logical_and(PASel01, PASel02)
#PASel = np.logical_and(PASel, np.logical_not(RID_Stat['MG CM Team']))

# if 'Latest Assigment Date (IMIS)' in NW_Ass_Case.columns.tolist():
#     pass
# else:
#     NW_Ass_Case['Latest Assigment Date (IMIS)'] = pd.NaT

tmpCol = NW_Ass_Case.columns.tolist()

tmpCol.remove('MasterGroup ID')
tmpCol.remove('Master_Group_Name')
if 'IMIS Group No' in tmpCol:
    tmpCol.remove('IMIS Group No')
#print(tmpCol)

RID_Stat['CM Assigned'] = ''
NW_Ass_Case_New_RID = RID_Stat[PASel][tmpCol]

SelNWMG = np.logical_not(NW_Ass_Case['IMISGroupNo'].isin(NW_Ass_Case_New_RID['IMISGroupNo']))
if np.sum(SelNWMG) > 0:
    NW_Ass_Case_New_RID = NW_Ass_Case_New_RID.append(NW_Ass_Case[SelNWMG], ignore_index=True)
NW_Ass_Case_New_RID = NW_Ass_Case_New_RID[np.logical_not(NW_Ass_Case_New_RID['IMISGroupNo'].isnull())]


    

    





In [139]:
Sel01 = CAListMst['CMB_Segment_S1'] == "BBRM"
Sel02 = CAListMst['IMIS Group No'] != ''
Sel = np.logical_and(Sel01, Sel02)
CARID_List = CAListMst[Sel]['IMIS Group No']

# Sel01 = NW_Ass_Case_New_RID['CM Assigned'].isnull()
# Sel02 = NW_Ass_Case_New_RID['MasterGroup ID'].isnull()
# Sel = np.logical_and(Sel01, Sel02)
# NWIPRID_List = NW_Ass_Case_New_RID[Sel]['IMISGroupNo'].tolist()


In [140]:
######## Lookup IMIS Group with preassigned CM ########

for aRID in CARID_List:
    SelRID = CAListMst['IMIS Group No'] == aRID
    
    if np.sum(SelRID) > 0:
        
        CMName = CAListMst[SelRID]['CM Name'].tolist()[0]
        if (CMName != '') & ((CMName[:3] != '***') & (CMName[:2] != 'CM')) :
            Sel = NW_Ass_Case_New_RID['IMISGroupNo'] == aRID
            NW_Ass_Case_New_RID.loc[Sel, 'CM Assigned'] = CMName

In [141]:
WIPMI_P = WIPMIRelID_P.append(WIPMIMG_P, ignore_index=True)
NW_Ass_Case_New = NW_Ass_Case_New_RID.append(NW_Ass_Case_New_MG, ignore_index=True)
now = datetime.now()
date_time = now.strftime("%Y%m%d%H%M%S")
NW_Ass_Case_New = NW_Ass_Case_New[NW_Ass_Case_New['Assign by TH'] != MGCSM_TH]
with pd.ExcelWriter('./Assigned MG without T0_' + date_time + '.xlsx') as WIPwriter:
    NW_Ass_Case_New.to_excel(WIPwriter, 'Non WIP Assigned Case', index=False)
    WIPMI_P.to_excel(WIPwriter, 'List of WIP Case', index=False)
    
print('PLease check the file %s to ensure if there is duplicated record.' % './Assigned MG without T0_' + date_time + '.xlsx')

PLease check the file ./Assigned MG without T0_ to ensure if there is duplicated record.20260817124927.xlsx


In [142]:
CheckSel01 = CAListMst['IMIS Group No'] != ''
CheckSel02 = CAListMst['CMB_Segment_S1'] == 'BBRM'
CheckSel = np.logical_and(CheckSel01, CheckSel02)
#CAListMst[CheckSel]['IMIS Group No'].unique()
IMISList = CAListMst[CheckSel]['IMIS Group No'].unique()

IMIS_Non_UniqueCMTH = pd.DataFrame(columns=['Type', 'Group ID'])

#print(IMIS_Non_UniqueCMTH)
ReqIMISID = []
for aID in IMISList:
    Sel01 = CAListMst['IMIS Group No'] == aID
    CMTH = CAListMst[Sel01]['Assign by TH'].unique()
    #print(CMTH.shape[0])
    if CMTH.shape[0] > 1:
        #print(CMTH)
        ReqIMISID.append(aID)

IMIS_Non_UniqueCMTH['Group ID'] = ReqIMISID
IMIS_Non_UniqueCMTH['Type'] = 'IMIS Group ID'

#for aID in IMISList:
for aID in ReqIMISID:
    Sel01 = CAListMst['IMIS Group No'] == aID
    CMTH = CAListMst[Sel01]['Assign by TH'].unique()
    Sel02 = IMIS_Non_UniqueCMTH['Group ID'] == aID
    
    for aCMTH in CMTH:
        if not aCMTH in IMIS_Non_UniqueCMTH.columns:
            IMIS_Non_UniqueCMTH[aCMTH] = ''
        SelCount = np.logical_and(Sel01, CAListMst['Assign by TH'] == aCMTH)
        IMIS_Non_UniqueCMTH.loc[Sel02, aCMTH] = np.sum(SelCount)
IMIS_Non_UniqueCMTH01 = IMIS_Non_UniqueCMTH
IMIS_Non_UniqueCMTH01

,Type,Group ID


In [143]:
try:
    CheckSel01 = CAListMst['Master Group ID'] != ''
except:
    try:
        CheckSel01 = CAListMst['Master Group ID'].replace(0, '') != ''
    except:
        CheckSel01 = CAListMst['Master Group ID'] > 0
CheckSel02 = (CAListMst['CMB_Segment_S1'] == 'CORP') | (CAListMst['CMB_Segment_S1'] == 'CMB')
CheckSel = np.logical_and(CheckSel01, CheckSel02)
#CAListMst[CheckSel]['IMIS Group No'].unique()
IMISList = CAListMst[CheckSel]['Master Group ID'].unique()

IMIS_Non_UniqueCMTH = pd.DataFrame(columns=['Type', 'Group ID'])

#print(IMIS_Non_UniqueCMTH)
ReqIMISID = []
for aID in IMISList:
    
    Sel01 = CAListMst['Master Group ID'] == aID
    CMTH = CAListMst[Sel01]['Assign by TH'].unique()
    #print(CMTH.shape[0])
    if CMTH.shape[0] > 1:
        #print(CMTH)
        ReqIMISID.append(aID)

IMIS_Non_UniqueCMTH['Group ID'] = ReqIMISID
IMIS_Non_UniqueCMTH['Type'] = 'Master Group ID'

#for aID in IMISList:
for aID in ReqIMISID:
    Sel01 = CAListMst['Master Group ID'] == aID
    CMTH = CAListMst[Sel01]['Assign by TH'].unique()
    Sel02 = IMIS_Non_UniqueCMTH['Group ID'] == aID
    
    for aCMTH in CMTH:
        if not aCMTH in IMIS_Non_UniqueCMTH.columns:
            IMIS_Non_UniqueCMTH[aCMTH] = ''
        SelCount = np.logical_and(Sel01, CAListMst['Assign by TH'] == aCMTH)
        IMIS_Non_UniqueCMTH.loc[Sel02, aCMTH] = np.sum(SelCount)
IMIS_Non_UniqueCMTH02 = IMIS_Non_UniqueCMTH
IMIS_Non_UniqueCMTH02

,Type,Group ID


In [144]:
NonUnique_CMTH = IMIS_Non_UniqueCMTH02.append(IMIS_Non_UniqueCMTH01, ignore_index=True).fillna('')

In [145]:
ColList = NonUnique_CMTH.columns.tolist()
ColList.remove('Group ID')
ColList.remove('Type')
ColList = ['Type', 'Group ID'] + ColList
NonUnique_CMTH = NonUnique_CMTH[ColList]
NonUnique_CMTH.to_excel('Client Group with more than 1 CMTH'  + date_time + '.xlsx', index=0)

### Split Result into DataFrame

In [146]:
CAListMst_Main_R = CAListMst[CAListMst['Customer_ID_S1'].isin(CAListMst_Main['Customer_ID_S1'])]

try:
    CAListMst_Main_R_Ext = CAListMst_Main_R[ColCAListMstKYC].copy(deep=True)
except KeyError:
    CAListMst_Main_R_Ext = CAListMst_Main_R[ColCAListMst].copy(deep=True)
CAListMst_Main_R_Ext.to_excel('./CM_Assign_Main.xlsx', index=False)

In [147]:
if FileName_Admin != '':
    CAListMst_Admin_R = CAListMst[CAListMst['Customer_ID_S1'].isin(CAListMst_Admin['Customer_ID_S1'])]
    
    try:
        CAListMst_Admin_R_Ext = CAListMst_Admin_R[ColCAListMstKYC].copy(deep=True)
    except KeyError:
        CAListMst_Admin_R_Ext = CAListMst_Admin_R[ColCAListMst].copy(deep=True)
    CAListMst_Admin_R_Ext.to_excel('./CM_Assign_Admin.xlsx', index=False)

In [148]:
if FileName_Amend != '':
    CAListMst_Amend_R = CAListMst[CAListMst['Customer_ID_S1'].isin(CAListMst_Amend['Customer_ID_S1'])]
    
    try:
        CAListMst_Amend_R_Ext = CAListMst_Amend_R[ColCAListMstKYC].copy(deep=True)
    except KeyError:
        CAListMst_Amend_R_Ext = CAListMst_Amend_R[ColCAListMst].copy(deep=True)
    CAListMst_Amend_R_Ext.to_excel('./CM_Assign_Amend.xlsx', index=False)




In [149]:
if FileName_FDD != '':
    CAListMst_FDD_R = CAListMst[CAListMst['Customer_ID_S1'].isin(CAListMst_FDD['Customer_ID_S1'])]
    
    try:
        CAListMst_FDD_R_Ext = CAListMst_FDD_R[ColCAListMstKYC].copy(deep=True)
    except KeyError:
        CAListMst_FDD_R_Ext = CAListMst_FDD_R[ColCAListMst].copy(deep=True)
    CAListMst_FDD_R_Ext.to_excel('./CM_Assign_FDD.xlsx', index=False)




### Copy Result to MS Excel FIle

In [150]:
def Copy_Result_To_Excel(MstExcelFName, ResultExcelFName, CMFullList_File, path):

    
    xcl = win32com.client.Dispatch('Excel.Application')
    xcl.visible = True
    xlsMst = xcl.workbooks.open(path + MstExcelFName.split('.')[0] + '.xlsx', 0, False, None)
    xlsRes = xcl.workbooks.open(path + ResultExcelFName, 0, False, None)
    xlsCM = xcl.workbooks.open(path + CMFullList_File, 0, False, None)
    MstWS = xlsMst.Worksheets(1)
    ResWS = xlsRes.Worksheets(1)
    CMWs = xlsCM.Worksheets(1)
    print(MstWS.name)
    ColIndx = xcl.WorksheetFunction.Match('CM Name', MstWS.Rows(1), 0)
    Col_HandleTeam01 = xcl.WorksheetFunction.Match('Case Handled by KYC Ops', MstWS.Rows(1), 0)
    Col_HandleTeam02 = xcl.WorksheetFunction.Match('Case Handled by KYC Ops', ResWS.Rows(1), 0)
    
    ResWS.Range(ResWS.Columns(1), ResWS.Columns(ColIndx)).Copy()
    MstWS.Range(MstWS.Columns(1), MstWS.Columns(ColIndx)).PasteSpecial(-4163)
    
    ResWS.Columns(Col_HandleTeam02).Copy()
    MstWS.Columns(Col_HandleTeam01).PasteSpecial(-4163)
    CMWs.Cells.Copy()
    xlsMst.Worksheets('CM List').Cells.PasteSpecial(-4163)
    
    
#     ColIndx = MstWS.Cells(1, 1).Value
#     print(ColIndx)
#     xcl.DisplayAlerts = False
    xlsMst.Save()
    xlsRes.Saved = True
    xlsCM.Saved = True
    xlsCM.Close()
    xlsMst.Close()
    xlsRes.Close()
#     xlsm.SaveAs(path + xlxs_FName.split('.')[0], 51, '')
#     xcl.DisplayAlerts = True
    xcl.Quit()
    return 0

In [151]:
Copy_Result_To_Excel(FName, 'CM_Assign_Main.xlsx', CMFullList_File, path)

CMAssignmentFullList


0

In [152]:
if FileName_Amend != '':
    Copy_Result_To_Excel(FileName_Amend, 'CM_Assign_Amend.xlsx', CMFullList_File, path)

In [153]:
if FileName_Admin != '':
    Copy_Result_To_Excel(FileName_Admin, 'CM_Assign_Admin.xlsx', CMFullList_File, path)

In [154]:
if FileName_FDD != '':
    Copy_Result_To_Excel(FileName_FDD, 'CM_Assign_FDD.xlsx', CMFullList_File, path)

### Split File into Onshore Team & Offshore Team

In [155]:
def Copy_Result_To_Excel(MstExcelFName, CAListMst_Src, path):
    Sel_OffShoreCase = CAListMst_Src['Case Handled by KYC Ops']
    TTL_OffshoreCase = np.sum(Sel_OffShoreCase)
    print('Number of Offshore Case:\t%d' % TTL_OffshoreCase)
    if TTL_OffshoreCase == 0:
        return -1
    elif (TTL_OffshoreCase == CAListMst_Src.shape[0]):
        os.rename(path + MstExcelFName.split('.')[0] + '.xlsx', path + MstExcelFName.split('.')[0] + ' - CDD Outreach Team' + '.xlsx')
        return 0
    else:
        
        
        
#         OffShoreCase = CAListMst_Src[Sel_OffShoreCase]['Customer_ID_S1'].tolist()
    
        OnShoreFName = MstExcelFName
        tmpName = MstExcelFName.split('.')
        OffShoreFName = tmpName[0] + ' - CDD Outreach Team' + '.xlsx'
        
        copyfile(path + MstExcelFName.split('.')[0] + '.xlsx', path + OffShoreFName)
        
        xcl = win32com.client.Dispatch('Excel.Application')
        xcl.visible = True
        
        ###### Handle Onshore Team
        
        xlsMst = xcl.workbooks.open(path + MstExcelFName.split('.')[0] + '.xlsx', 0, False, None)
        MstWS = xlsMst.Worksheets(1)
        
        CustIDColRef = xcl.WorksheetFunction.Match('Customer_ID_S1', MstWS.Rows(1), 0)
        FilterCol = xcl.WorksheetFunction.Match('Case Handled by KYC Ops', MstWS.Rows(1), 0)
#         print(FilterCol)
        
        MstWS.Select()
        MstWS.Range("A1").Select()
        LastRow = MstWS.Range("A1048576").End(-4162).Row
        MstWS.Range("A1").AutoFilter(FilterCol, "TRUE")
#         LastRow = MstWS.Range("A1048576").End(-4162).Row
        MstWS.Range("A1").Select()
#         FirstRow = min(CAListMst[CAListMst['Case Handled by KYC Ops']]['Org_Seq'].index.tolist()) + 2
        FirstRow = 2
        print('First Row:\t%d' % FirstRow)
        print('Last Row:\t%d' % LastRow)
        MstWS.Rows(str(FirstRow) + ":" + str(LastRow)).Delete()
        MstWS.AutoFilter.ShowAllData()
        print('No of Case (Onshore):\t%d' % int(MstWS.Range("A1048576").End(-4162).Row - 1))
        xlsMst.Save()
        xlsMst.Close()
        
        ###### Handle Offshore Team
        
        xlsMst = xcl.workbooks.open(path + OffShoreFName, 0, False, None)
        MstWS = xlsMst.Worksheets(1)
        
        CustIDColRef = xcl.WorksheetFunction.Match('Customer_ID_S1', MstWS.Rows(1), 0)
        FilterCol = xcl.WorksheetFunction.Match('Case Handled by KYC Ops', MstWS.Rows(1), 0)
#         print(FilterCol)
        MstWS.Select()
        MstWS.Range("A1").Select()
        LastRow = MstWS.Range("A1048576").End(-4162).Row
        MstWS.Range("A1").AutoFilter(FilterCol, "FALSE")
#         LastRow = MstWS.Range("A1048576").End(-4162).Row
        MstWS.Range("A1").Select()
#         FirstRow = min(CAListMst[~CAListMst['Case Handled by KYC Ops']]['Org_Seq'].index.tolist()) + 2
        FirstRow = 2
        print('First Row:\t%d' % FirstRow)
        print('Last Row:\t%d' % LastRow)
        
#         MstWS.Rows(str(FirstRow) + ":" + str(FirstRow)).Select()
        MstWS.Rows(str(FirstRow) + ":" + str(LastRow)).Delete()
        MstWS.AutoFilter.ShowAllData()
        
        
        print('No of Case (Offshore):\t%d' % int(MstWS.Range("A1048576").End(-4162).Row - 1))
        xlsMst.Save()
        xlsMst.Close()
        
        xcl.Quit()
#         for aID in OffShoreCase:
#             try:
#                 RecordRow = xcl.WorksheetFunction.Match(aID, MstWS.Cells(1, CustIDColRef).EntireColumn, 0)
#                 MstWS.Rows(RecordRow).Delete()
#                 CountDel =+ 1
# #                 print('Customer ID:\t%s \t@\t%d' % (aID, RecordRow))
#             except:
#                 pass
# #                 print(aID)
        return 1

In [156]:
%%time
if KYC_CM_Exist:
    Copy_Result_To_Excel(FName, CAListMst_Main_R, path)
    if FileName_Amend != '':
        Copy_Result_To_Excel(FileName_Amend, CAListMst_Amend_R, path)
    if FileName_Admin != '':
        Copy_Result_To_Excel(FileName_Admin, CAListMst_Admin_R, path)
    if FileName_FDD != '':
        Copy_Result_To_Excel(FileName_FDD, CAListMst_FDD_R, path)

CPU times: total: 0 ns
Wall time: 0 ns


# Visualization

In [157]:
# CMTeam

In [158]:
# CMTeam_New = CMTeam.copy(deep=True)

# ttlBBRM = np.sum(CAListMst['CMB_Segment_S1'] == 'BBRM')
# ttlCorp = np.sum(CAListMst['CMB_Segment_S1'] == 'CORP')

# CMTeam_New['Proposed No of BBRM'] = np.ceil(CMTeam_New['% HP'] * ttlBBRM)
# CMTeam_New['Proposed No of CORP'] = np.ceil(CMTeam_New['% HP'] * ttlCorp)
# CMTeam_New['Diff No of CORP'] = CMTeam_New['Total No of Corp Assigned'] - CMTeam_New['Proposed No of CORP']
# CMTeam_New['Diff No of BBRM'] = CMTeam_New['Total No of BBRM Assigned'] - CMTeam_New['Proposed No of BBRM']
# UnEvenIndex_Corp = np.sum(np.absolute(CMTeam_New['Diff No of CORP']))
# UnEvenIndex_BBRM = np.sum(np.absolute(CMTeam_New['Diff No of BBRM']))
# UnEvenIndex = np.sum(np.absolute(CMTeam_New['Diff No of CORP'])) + np.sum(np.absolute(CMTeam_New['Diff No of BBRM']))
                          
# print('UnEven Index (Corp Team):\t%d' % UnEvenIndex_Corp)
# print('UnEven Index (BBRM Team):\t%d' % UnEvenIndex_BBRM)
# print('UnEven Index (Overall):\t%d' % UnEvenIndex)
# CMTeam_New

In [159]:
# AllIndex = CMPort.index.tolist()

# AllBBRM = []
# AllCorp = []

# for aInd in AllIndex:
#     if aInd[0] == 'BBRM':
#         AllBBRM.append(aInd[1])
#         pass
#     else:
#         AllCorp.append(aInd[1])

In [160]:
# import matplotlib.pyplot as plt; plt.rcdefaults()
# import matplotlib.pyplot as plt

In [161]:
# AllCMTH = CMPort.columns.tolist()
# for CMTH in AllCMTH:
    
    
#     plt.close
#     objects = CMPort.index.get_level_values(1).tolist()
#     y_pos = range(1, len(CMPort.index.tolist())+1)
#     plt.figure(figsize=(15, 10))
#     plt.bar(y_pos, CMPort[CMTH].values, align='center', alpha=0.8)
#     plt.xticks(y_pos, objects)
#     plt.ylabel('Usage')
#     plt.axvline(x=9.5, color='green')
#     plt.grid()
#     plt.title('Case Assignment for ' + str(CMTH[1]))

#     plt.show()

In [162]:
# pd.set_option('display.max_columns', None)

# Version Control

**This file was updated on 6 Jun 2022.**<br>
Update code for Case Assignment for MGCSM assign CM to wrong TH.<br>

**This file was updated on 2 Jul 2021.**<br>
Update code for Case Assignment for Corp cases only.<br>

**This file was updated on 18 May 2021.**<br>
Minor debug for Last Assignment Date for easy reference in the Assigned MG without T0 File.<br>

**This file was updated on 5 May 2021.**<br>
Added Last Assignment Date for easy reference in the Assigned MG without T0 File.<br>

**This file was updated on 28 Apr 2021.**<br>
Disable coding for Onshore / Offshore different treatment.<br>
Update code for Corp case to assign to CM Level for WIP cases.<br>

**This file was updated on 10 Mar 2021.**<br>
Update Code for Splitting Files for Onshore and Offshore Team.<br>

**This file was updated on 18 Feb 2021.**<br>
Update Code for copying CM List.<br>

**This file was updated on 16 Feb 2021.**<br>
Update Code for copying result.<br>

**This file was updated on 10 Feb 2021.**<br>
Update Code for opening xlsm version.<br>

**This file was updated on 26 Nov 2020.**<br>
Code fixing for Pandas 0.23.<br>